In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
import warnings
from typing import TypedDict, Literal, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import subprocess
from typing import TypedDict, Literal, Any, Annotated
import operator



pd.set_option('display.max_columns', 100)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


## Обьявляем все ЛЛМ

### Описание

In [4]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'openai/gpt-4.1-mini'
model = 'google/gemini-3-flash-preview'

llm_descr = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=5000
)

In [5]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import io
import contextlib
import traceback
from pathlib import Path


class PythonCodeInput(BaseModel):
    code: str = Field(description="Python code to execute.")


@tool(args_schema=PythonCodeInput)
def python_interpreter_tool(code: str) -> dict[str, Any]:
    """
    Executes Python code in isolated local memory.
    Does not return variables or dataframe previews.
    """

    forbidden_patterns = [
        "rm -rf",
        "shutil.rmtree",
        "os.remove",
        "os.rmdir",
        "subprocess",
        "socket",
        "requests.",
        "urllib",
    ]

    lowered_code = code.lower()

    if any(pattern.lower() in lowered_code for pattern in forbidden_patterns):
        return {
            "status": "failed",
            "stdout": "",
            "error": "Code rejected by safety filter.",
        }

    local_memory = {}

    stdout_buffer = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, local_memory, local_memory)

        return {
            "status": "success",
            "stdout": stdout_buffer.getvalue()[:5000],
            "error": None,
        }

    except Exception:
        return {
            "status": "failed",
            "stdout": stdout_buffer.getvalue()[:5000],
            "error": traceback.format_exc(),
        }

In [6]:
from langchain.agents import create_agent

DATA_DESCRIPTION_STAGE_PROMPT = """
You are a Data Description Agent in a multi-agent ML system.

You analyze datasets step by step.

Rules:
- Use python_interpreter_tool when code execution is needed.
- Write short Python code only for the current stage.
- Do not solve all EDA tasks at once.
- Do not preprocess data.
- Do not train models.
- Do not modify the original dataset.
- Do not use internet or APIs.
- Use only pandas, numpy, json, pathlib, re, collections.
- Return only valid JSON for the current stage.
- Do not include Python code in the final answer.
- Do not include tool calls, tool code, stdout, or debug output in the final answer.
- Do not use markdown.
- Do not add explanations outside JSON.
"""


data_description_stage_agent = create_agent(
    model=llm_descr,
    tools=[python_interpreter_tool],
    system_prompt=DATA_DESCRIPTION_STAGE_PROMPT,
)

### Подготовка данных

In [7]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'openai/gpt-4.1-mini'
model = 'google/gemini-3-flash-preview'
# model = 'openai/gpt-5.4-mini'

llm_prep = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=5000
)

In [8]:
from langchain.agents import create_agent

DATA_PREPARATION_STAGE_PROMPT = """
You are a Data Preparation Agent in a multi-agent ML system.
You prepare datasets step by step according to the current stage requested by the orchestrator.
Rules:
- Use python_interpreter_tool when code execution is needed.
- Write short Python code only for the current stage.
- Do not solve multiple pipeline stages at once.
- Do not perform full EDA.
- Do not train models.
- Do not modify the original dataset.
- Do not use internet or APIs.
- Use only pandas, numpy, json, pathlib, re, collections.
- Use the schema provided by the orchestrator as the source of truth.
- Do not redefine feature types unless the current stage explicitly asks you to validate them.
- Do not use the target column for feature engineering.
- Save intermediate datasets only to the path requested by the orchestrator.
- Return only valid JSON for the current stage.
- Do not include Python code in the final answer.
- Do not include tool calls, tool code, stdout, or debug output in the final answer.
- Do not use markdown.
- Do not add explanations outside JSON."""

data_preparation_stage_agent = create_agent(
    model=llm_prep,
    tools=[python_interpreter_tool],
    system_prompt=DATA_PREPARATION_STAGE_PROMPT,
)

### Обучатор моделей

In [9]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'openai/gpt-4.1-mini'
model = 'google/gemini-3-flash-preview'
# model = 'openai/gpt-5.4-mini'

llm_model = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=5000
)

In [10]:
MODELING_STAGE_PROMPT = """
You are a Modeling Agent in a multi-agent ML system.

You train and evaluate machine learning models step by step according to the current stage requested by the orchestrator.

Rules:
- Use python_interpreter_tool when code execution is needed.
- Write short Python code only for the current stage.
- Do not solve multiple pipeline stages at once.
- Do not perform data preparation.
- Do not modify the original prepared dataset.
- Do not use internet or APIs.
- Use only pandas, numpy, json, pathlib, sklearn, joblib.
- Use the prepared dataset path provided by the orchestrator.
- Use the target column provided by the orchestrator.
- Use the task type provided by the orchestrator.
- Do not use the target column as a feature.
- Save model artifacts only to the path requested by the orchestrator.
- Return only valid JSON for the current stage.
- Do not include Python code in the final answer.
- Do not include tool calls, tool code, stdout, or debug output in the final answer.
- Do not use markdown.
- Do not add explanations outside JSON.
"""

modeling_stage_agent = create_agent(
    model=llm_model,
    tools=[python_interpreter_tool],
    system_prompt=MODELING_STAGE_PROMPT,
)

### Сообщатор

In [11]:
from langchain.agents import create_agent

REPORTING_AGENT_PROMPT = """
You are a Reporting Agent in a multi-agent ML system.

Your task is to create a final project report based on results from previous agents.

Rules:
- Use python_interpreter_tool to write the report file.
- Do not train models.
- Do not modify datasets.
- Do not perform data preparation.
- Do not use internet or APIs.
- Use only json, pathlib, pandas if needed.
- Write the report in clear Markdown.
- Include both technical and business interpretation.
- Use only provided summaries and artifacts.
- Do not invent metrics.
- If some information is missing, write "not available".
- Return only valid JSON.
- Do not include Python code in the final answer.
- Do not include tool calls, tool code, stdout, or debug output in the final answer.
- Do not use markdown outside the report file.
"""

In [12]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'openai/gpt-4.1-mini'
model = 'google/gemini-3-flash-preview'
# model = 'openai/gpt-5.4-mini'

llm_reporting = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=5000
)
reporting_agent = create_agent(
    model=llm_reporting,
    tools=[python_interpreter_tool],
    system_prompt=REPORTING_AGENT_PROMPT,
)

### Оркестратор

In [13]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'google/gemini-3-flash-preview'
# model = 'openai/gpt-5.4-mini'

llm_orchestrator = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=3000
)

## Собираем оркестратор

In [14]:

class AgentState(TypedDict, total=False):
    run_id: str | None

    dataset_path: str | None
    target_column: str | None
    task_type: Literal["classification", "regression", "auto"]

    schema: dict[str, list[str]]
    data_summary: dict[str, Any]
    prep_summary: dict[str, Any]
    text_summary: dict[str, Any]
    modeling_summary: dict[str, Any]

    artifacts: dict[str, str | None]
    logs: Annotated[list[dict[str, Any]], operator.add]

    current_step: str | None
    next_action: str | None
    orchestration_decision: dict[str, Any]
    orchestration_history: Annotated[list[dict[str, Any]], operator.add]

    previous_run: dict[str, Any]
    previous_best_model: dict[str, Any]

    retry_count: int
    max_retries: int
    status: str
    error: str | None

In [15]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are an LLM-based ML Workflow Orchestrator Agent.

Your role:
- analyze the current AgentState
- decide what should happen next
- choose which specialized agent should be called
- explain the reason for your decision
- handle errors, retries, fallback logic, and finishing conditions
- use available tools when filesystem inspection is needed
- find the input dataset path when it is missing
- inspect artifact folders to recover previous run information

You do NOT train models directly.
You do NOT preprocess data directly.
You do NOT create reports directly.
You only orchestrate the workflow.

Available tools:
- bash_tool: use it only for safe filesystem inspection, such as finding CSV files, listing artifact folders, or reading JSON metadata.

Available decisions:
- run_data_description
- run_tabular_preparation
- run_text_preparation
- merge_features
- run_modeling
- run_improvement
- run_reporting
- retry_current_step
- use_fallback
- finish
- fail

Specialized agents:
- data_description_agent
- tabular_preparation_agent
- text_preparation_agent
- merge_features
- modeling_agent
- improvement_agent
- reporting_agent

A step is completed ONLY by the following exact rules:

1. data_description_agent is completed if:
   - data_summary contains key "n_rows"
   - data_summary contains key "n_columns"
   - schema contains key "numeric"
   - schema contains key "categorical"
   - schema contains key "text"
   - schema contains key "datetime"

2. tabular_preparation_agent is completed if:
   - prep_summary contains key "missing_strategy"
   - prep_summary contains key "encoding"
   - prep_summary contains key "scaling"
   - prep_summary contains key "created_features"

3. text_preparation_agent is completed if:
   - schema["text"] is an empty list
   OR
   - text_summary["used"] is true

4. merge_features is completed if:
   - artifacts contains key "prepared_dataset_path"
   - artifacts["prepared_dataset_path"] is not null

5. modeling_agent is completed if:
   - modeling_summary contains key "best_model_name"
   - modeling_summary["best_model_name"] is not null

6. reporting_agent is completed if:
   - artifacts contains key "final_report_path"
   - artifacts["final_report_path"] is not null

Decision order:
1. If error is not null and retry_count < max_retries, choose retry_current_step.
2. If error is not null and retry_count >= max_retries, choose use_fallback or fail.
3. If data_description_agent is not completed, choose run_data_description.
4. Else if tabular_preparation_agent is not completed, choose run_tabular_preparation.
5. Else if text_preparation_agent is not completed, choose run_text_preparation.
6. Else if merge_features is not completed, choose merge_features.
7. Else if modeling_agent is not completed, choose run_modeling.
8. Else if modeling_summary["retry_recommended"] is true and retry_count < max_retries, choose run_improvement.
9. Else if reporting_agent is not completed, choose run_reporting.
10. Else choose finish.

Strict rules:
- Do not invent additional completion requirements.
- Do not say a step is "not fully populated" if the exact completion keys exist.
- Do not choose run_data_description if data_summary has n_rows and n_columns and schema has numeric, categorical, text, datetime.
- Do not choose run_tabular_preparation if prep_summary has missing_strategy, encoding, scaling, created_features.
- Do not choose run_modeling before merge_features is completed.
- tabular_dataset_path is not the same as prepared_dataset_path.
- Modeling can start only after prepared_dataset_path exists.
- Do not choose retry_current_step if error is null.
- Do not choose use_fallback if error is null.
- Use bash_tool only for safe filesystem inspection.
- Do not delete, move, overwrite, or modify files using bash_tool.
- For dataset search, prefer CSV files inside ./data.
- Ignore CSV files inside ./artifacts.
- Ignore predictions.csv, metrics.csv, and temporary output CSV files.
- Return only valid JSON when a structured decision is requested.
"""

In [16]:
initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "price",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

### Первые ноды для поиска датсета и предыдущего рана

In [17]:
class BashInput(BaseModel):
    command: str = Field(
        description="Bash command to execute."
    )

@tool(args_schema=BashInput)
def bash_tool(command: str) -> dict[str, Any]:
    """
    Executes a bash command and returns stdout, stderr and return code.
    Use it for filesystem operations:
    - finding csv files
    - listing artifact folders
    - reading json files
    """

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )

        return {
            "status": "success" if result.returncode == 0 else "failed",
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
            "returncode": result.returncode,
        }

    except Exception as e:
        return {
            "status": "failed",
            "stdout": "",
            "stderr": str(e),
            "returncode": -1,
        }

In [18]:
from langchain.agents import create_agent

tools = [bash_tool]

orchestrator_tool_agent = create_agent(
    model=llm_orchestrator,
    tools=tools,
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
)

In [19]:
import re


def extract_csv_path(text: str) -> str | None:
    match = re.search(r"([./\w\-/]+\.csv)", text)
    if match:
        return match.group(1)
    return None


def dataset_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the main CSV dataset in the current project folder. "
                    "Use bash_tool if needed. "
                    "Prefer files inside ./data. "
                    "Ignore files inside ./artifacts. "
                    "Ignore predictions.csv, metrics.csv, and temporary output files. "
                    "Return only the dataset path as plain text. "
                    "Do not add explanations."
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()

    dataset_path = extract_csv_path(agent_response)

    # fallback через bash, если агент ответил криво
    fallback_used = False
    fallback_result = None

    if not dataset_path:
        fallback_used = True
        fallback_result = bash_tool.invoke({
            "command": (
                'find . -type f -name "*.csv" '
                '! -path "./artifacts/*" '
                '! -name "predictions.csv" '
                '! -name "metrics.csv" '
                '2>/dev/null | head -n 1'
            )
        })

        dataset_path = fallback_result.get("stdout", "").strip()
        if dataset_path:
            dataset_path = dataset_path.splitlines()[0].strip()

    if not dataset_path:
        return {
            "error": f"Dataset path was not found. Agent returned: {agent_response}",
            "logs": [{
                "agent": "dataset_finder",
                "status": "failed",
                "skipped": False,
                "summary": "Dataset finder failed to locate a valid CSV file.",
                "decisions": {
                    "agent_response": agent_response,
                    "fallback_used": fallback_used,
                    "fallback_result": fallback_result,
                },
                "artifacts": {},
                "next_input": {},
                "reason": "No valid CSV path found.",
            }],
        }

    return {
        "dataset_path": dataset_path,
        "logs": [{
            "agent": "dataset_finder",
            "status": "success",
            "skipped": False,
            "summary": f"Dataset found: {dataset_path}",
            "decisions": {
                "agent_response": agent_response,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "dataset_path": dataset_path,
            },
            "reason": None,
        }],
    }

In [20]:
import json
import re


def extract_json_safe(text: str) -> dict | None:
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None


def previous_run_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the latest previous ML run inside the artifacts folder. "
                    "Use bash_tool if needed. "
                    "Look for run directories that may contain metrics.json and best_model.pkl. "
                    "Return only valid JSON, no markdown, no explanations. "
                    "JSON format:\n"
                    "{\n"
                    '  "latest_run_path": "path or null",\n'
                    '  "metrics_path": "path or null",\n'
                    '  "best_model_path": "path or null"\n'
                    "}"
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()
    parsed = extract_json_safe(agent_response)

    # Fallback: если LLM не вернула нормальный JSON,
    # сами через bash ищем последний run.
    fallback_used = False
    fallback_result = None

    if parsed is None:
        fallback_used = True

        fallback_result = bash_tool.invoke({
            "command": (
                'latest_run=$(ls -td artifacts/runs/* 2>/dev/null | head -n 1); '
                'if [ -z "$latest_run" ]; then '
                'echo \'{"latest_run_path": null, "metrics_path": null, "best_model_path": null}\'; '
                'else '
                'metrics_path="$latest_run/metrics.json"; '
                'model_path="$latest_run/best_model.pkl"; '
                'if [ ! -f "$metrics_path" ]; then metrics_path=null; fi; '
                'if [ ! -f "$model_path" ]; then model_path=null; fi; '
                'echo "{\\"latest_run_path\\": \\"$latest_run\\", \\"metrics_path\\": \\"$metrics_path\\", \\"best_model_path\\": \\"$model_path\\"}"; '
                'fi'
            )
        })

        parsed = extract_json_safe(fallback_result.get("stdout", ""))

    # Если даже fallback не сработал — считаем, что прошлого run нет.
    if parsed is None:
        parsed = {
            "latest_run_path": None,
            "metrics_path": None,
            "best_model_path": None,
        }

    latest_run_path = parsed.get("latest_run_path")
    metrics_path = parsed.get("metrics_path")
    best_model_path = parsed.get("best_model_path")

    if latest_run_path in [None, "null", "None", ""]:
        previous_run = {
            "exists": False,
            "path": None,
        }

        previous_best_model = {}

        log_record = {
            "agent": "previous_run_finder",
            "status": "success",
            "skipped": True,
            "summary": "No previous run was found.",
            "decisions": {
                "agent_response": agent_response,
                "parsed": parsed,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "previous_run": previous_run,
                "previous_best_model": previous_best_model,
            },
            "reason": "No previous run directory exists.",
        }

        return {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
            "logs": [log_record],
        }

    previous_metrics = {}

    if metrics_path not in [None, "null", "None", ""]:
        metrics_result = bash_tool.invoke({
            "command": f'cat "{metrics_path}" 2>/dev/null'
        })

        metrics_raw = metrics_result.get("stdout", "").strip()

        if metrics_raw:
            try:
                previous_metrics = json.loads(metrics_raw)
            except json.JSONDecodeError:
                previous_metrics = {
                    "raw_metrics": metrics_raw,
                }

    previous_run = {
        "exists": True,
        "path": latest_run_path,
        "metrics_path": metrics_path,
        "best_model_path": best_model_path,
    }

    previous_best_model = {
        "run_path": latest_run_path,
        "model_path": best_model_path,
        "metrics_path": metrics_path,
        "metrics": previous_metrics,
    }

    log_record = {
        "agent": "previous_run_finder",
        "status": "success",
        "skipped": False,
        "summary": f"Previous run found: {latest_run_path}",
        "decisions": {
            "agent_response": agent_response,
            "parsed": parsed,
            "fallback_used": fallback_used,
            "fallback_result": fallback_result,
            "previous_metrics": previous_metrics,
        },
        "artifacts": {},
        "next_input": {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
        },
        "reason": None,
    }

    return {
        "previous_run": previous_run,
        "previous_best_model": previous_best_model,
        "logs": [log_record],
    }

In [21]:
def join_init_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "join_init",
        "status": "success",
        "skipped": False,
        "summary": "Initialization steps completed: dataset search and previous run search are done.",
        "decisions": {
            "dataset_path": state.get("dataset_path"),
            "previous_run": state.get("previous_run", {}),
            "previous_best_model": state.get("previous_best_model", {}),
        },
        "artifacts": {},
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "join_init",
        "logs": [log_record],
    }

### Нода для выбора некст шага

In [22]:
from typing import Literal, Optional, Any
from pydantic import BaseModel, Field


class OrchestratorDecision(BaseModel):
    decision: Literal[
        "run_data_description",
        "run_tabular_preparation",
        "run_text_preparation",
        "merge_features",
        "run_modeling",
        "run_improvement",
        "run_reporting",
        "retry_current_step",
        "use_fallback",
        "finish",
        "fail",
    ] = Field(
        description="Следующее действие, которое выбрал orchestrator."
    )

    reason: str = Field(
        description="Краткое объяснение, почему выбрано именно это действие."
    )

    selected_agent: Optional[str] = Field(
        default=None,
        description="Название агента, которого нужно вызвать следующим."
    )

    input_needed: dict[str, Any] = Field(
        default_factory=dict,
        description="Какие входные данные нужны следующему агенту."
    )

    expected_output: list[str] = Field(
        default_factory=list,
        description="Какие результаты ожидаются от следующего агента."
    )

    risk_notes: list[str] = Field(
        default_factory=list,
        description="Риски или важные замечания по текущему шагу."
    )

In [23]:
def route_by_orchestrator_decision(state: AgentState) -> str:
    next_action = state.get("next_action")

    schema = state.get("schema", {})
    data_summary = state.get("data_summary", {})
    data_preparation = state.get("data_preparation", {})
    modeling_summary = state.get("modeling_summary", {})
    artifacts = state.get("artifacts", {})

    data_done = (
        "n_rows" in data_summary
        and "n_columns" in data_summary
        and all(
            k in schema
            for k in [
                "numeric",
                "categorical",
                "text",
                "datetime",
            ]
        )
    )

    prep_done = (
        artifacts.get("prepared_dataset_path") is not None
        or data_preparation.get("status") == "success"
    )

    modeling_done = (
        modeling_summary.get("best_model_name") is not None
    )

    reporting_done = (
        artifacts.get("final_report_path") is not None
    )

    error_exists = state.get("error") is not None

    legacy_mapping = {
        "run_tabular_preparation": "run_data_preparation",
        "run_text_preparation": "run_data_preparation",
        "merge_features": "run_data_preparation",
    }

    if next_action in legacy_mapping:
        next_action = legacy_mapping[next_action]

    # 1. Если LLM не выбрала действие
    if next_action is None:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_data_preparation"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 2. Retry/fallback разрешены только при реальной ошибке
    if next_action in ["retry_current_step", "use_fallback"] and not error_exists:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_data_preparation"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 3. Если есть ошибка — можно доверить LLM retry/fallback/fail
    if error_exists:
        if next_action in ["retry_current_step", "use_fallback", "fail"]:
            return next_action
        return "retry_current_step"

    # 4. Нельзя повторять data_description, если он уже выполнен
    if next_action == "run_data_description" and data_done:
        if not prep_done:
            return "run_data_preparation"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 5. Нельзя запускать data_preparation до data_description
    if next_action == "run_data_preparation" and not data_done:
        return "run_data_description"

    # 6. Нельзя повторять data_preparation, если prepared_dataset уже есть
    if next_action == "run_data_preparation" and prep_done:
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 7. Нельзя запускать modeling до data_preparation
    if next_action == "run_modeling" and not prep_done:
        if not data_done:
            return "run_data_description"
        return "run_data_preparation"

    # 8. Нельзя повторять modeling, если модель уже обучена
    if next_action == "run_modeling" and modeling_done:
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 9. Improvement можно запускать только после modeling
    if next_action == "run_improvement" and not modeling_done:
        if not prep_done:
            if not data_done:
                return "run_data_description"
            return "run_data_preparation"
        return "run_modeling"

    # 10. Reporting можно запускать только после modeling
    if next_action == "run_reporting" and not modeling_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_data_preparation"
        return "run_modeling"

    # 11. Нельзя повторять reporting, если отчет уже создан
    if next_action == "run_reporting" and reporting_done:
        return "finish"

    # 12. Нельзя завершать workflow до создания отчета
    if next_action == "finish" and not reporting_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_data_preparation"
        if not modeling_done:
            return "run_modeling"
        return "run_reporting"

    # 13. Если отчет есть, но status еще не success, отправляем в reporting
    if next_action == "finish" and reporting_done and state.get("status") != "success":
        return "run_reporting"

    # 14. Если LLM вдруг снова вернула старое действие
    if next_action in legacy_mapping:
        return legacy_mapping[next_action]

    # 15. Иначе доверяем решению LLM
    return next_action

In [24]:
import json
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import ValidationError


import json
import re


def extract_json(text: str) -> dict:
    """
    Достает JSON из ответа модели.
    Работает, даже если модель добавила thought, markdown или текст до/после JSON.
    """
    if text is None:
        raise ValueError("Empty response: text is None")

    text = text.strip()

    if not text:
        raise ValueError("Empty response: text is empty")

    # убираем markdown-обертку
    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    # пробуем распарсить как есть
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # если модель написала thought перед JSON — берем от первой { до последней }
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        json_text = text[start:end + 1]
        return json.loads(json_text)

    raise ValueError(f"Could not extract JSON from response. First 500 chars: {text[:500]}")


def call_orchestrator_llm_orchestrator(state: dict) -> OrchestratorDecision:
    """
    Sends current AgentState to the llm_orchestrator orchestrator
    and returns a validated OrchestratorDecision.
    """

    state_for_llm_orchestrator = {
        "run_id": state.get("run_id"),
        "dataset_path": state.get("dataset_path"),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),

        "schema": state.get("schema", {}),
        "data_summary": state.get("data_summary", {}),
        "prep_summary": state.get("prep_summary", {}),
        "text_summary": state.get("text_summary", {}),
        "modeling_summary": state.get("modeling_summary", {}),
        "artifacts": state.get("artifacts", {}),

        "available_keys": {
            "schema_keys": list(state.get("schema", {}).keys()),
            "data_summary_keys": list(state.get("data_summary", {}).keys()),
            "prep_summary_keys": list(state.get("prep_summary", {}).keys()),
            "text_summary_keys": list(state.get("text_summary", {}).keys()),
            "modeling_summary_keys": list(state.get("modeling_summary", {}).keys()),
            "artifact_keys": list(state.get("artifacts", {}).keys()),
        },

        "logs_count": len(state.get("logs", [])),
        "current_step": state.get("current_step"),
        "next_action": state.get("next_action"),
        "orchestration_history": state.get("orchestration_history", [])[-5:],

        "retry_count": state.get("retry_count", 0),
        "max_retries": state.get("max_retries", 1),
        "status": state.get("status"),
        "error": state.get("error"),
    }

    response = llm_orchestrator.invoke(
        [
            SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    "Analyze the current AgentState and choose the next action.\n\n"
                    "Return ONLY valid JSON. Do not use markdown. Do not add explanations outside JSON.\n\n"
                    "JSON format:\n"
                    "{\n"
                    '  "decision": "run_data_description",\n'
                    '  "reason": "short reason",\n'
                    '  "selected_agent": "data_description_agent",\n'
                    '  "input_needed": {},\n'
                    '  "expected_output": [],\n'
                    '  "risk_notes": []\n'
                    "}\n\n"
                    f"AgentState:\n{json.dumps(state_for_llm_orchestrator, ensure_ascii=False, indent=2)}"
                )
            ),
        ]
    )

    raw_text = response.content
    parsed = extract_json(raw_text)

    try:
        return OrchestratorDecision(**parsed)
    except ValidationError as e:
        raise ValueError(
            f"llm_orchestrator returned invalid OrchestratorDecision.\n"
            f"Validation error: {e}\n"
            f"Raw response: {raw_text}"
        )

In [25]:
def orchestrator_agent_node(state: AgentState) -> AgentState:
    decision = call_orchestrator_llm_orchestrator(state)
    decision_dict = decision.model_dump()

    log_record = {
        "agent": "orchestrator_agent",
        "status": "success",
        "skipped": False,
        "summary": decision.reason,
        "decisions": decision_dict,
        "artifacts": {},
        "next_input": decision.input_needed,
        "reason": None,
    }

    return {
        "current_step": "orchestrator_agent",
        "next_action": decision.decision,
        "orchestration_decision": decision_dict,
        "orchestration_history": [decision_dict],
        "logs": [log_record],
    }

## Добавляем агента для описания

In [26]:
from typing import TypedDict, Any, Annotated
import operator


class DataDescriptionState(TypedDict, total=False):
    dataset_path: str
    target_column: str | None
    artifacts_dir: str

    basic_overview: dict[str, Any]
    domain_understanding: dict[str, Any]
    schema_detection: dict[str, Any]
    data_quality: dict[str, Any]
    statistics: dict[str, Any]

    final_result: dict[str, Any]
    artifacts: dict[str, str]

    logs: Annotated[list[dict[str, Any]], operator.add]
    error: str | None

In [27]:
def run_stage_with_llm(stage_name: str, task: str) -> dict:
    print(f"\n========== START STAGE: {stage_name} ==========")

    try:
        result = data_description_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise


In [28]:
from pathlib import Path
import json


def dd_basic_overview_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]

    task = f"""
Stage: basic_overview

Read the training dataset from:
{dataset_path}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Calculate:
   - n_rows
   - n_columns
   - columns
   - dtypes
   - missing_values_total
   - duplicate_rows
3. Do not print Python code or debug output.

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Each later stage will read the dataset again if needed.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "basic_overview",
  "status": "success",
  "result": {{
    "n_rows": 0,
    "n_columns": 0,
    "columns": [],
    "dtypes": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("basic_overview", task)
        basic_overview = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        basic_overview_path = artifacts_dir / "basic_overview.json"

        with open(basic_overview_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "basic_overview": basic_overview,
            "artifacts": {
                **state.get("artifacts", {}),
                "basic_overview_path": str(basic_overview_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "basic_overview",
                "status": "success",
                "summary": "Basic dataset overview completed.",
                "artifacts": {
                    "basic_overview_path": str(basic_overview_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "basic_overview",
                "status": "failed",
                "summary": "Basic overview failed.",
                "reason": str(e),
            }],
        }

In [29]:
from pathlib import Path
import json


def dd_domain_understanding_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    basic = state.get("basic_overview", {})

    task = f"""
Stage: domain_understanding

Read the training dataset from:
{dataset_path}

Previous result:
{json.dumps(basic, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Inspect only compact samples:
   - df.head(3)
   - column names
   - sample values from important object/string columns
3. Do not print Python code or debug output.
4. Do not print the full dataframe.
5. Do not keep the full dataframe in memory for next stages.

Infer:
- subject area of the dataset
- what one row represents
- what groups of columns describe

Important rules:
- Do not preprocess data.
- Do not modify df.
- Use column names and sample values to infer the domain.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "domain_understanding",
  "status": "success",
  "result": {{
    "domain_summary": "...",
    "row_meaning": "...",
    "column_groups_description": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("domain_understanding", task)
        domain_understanding = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        domain_understanding_path = artifacts_dir / "domain_understanding.json"

        with open(domain_understanding_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "domain_understanding": domain_understanding,
            "artifacts": {
                **state.get("artifacts", {}),
                "domain_understanding_path": str(domain_understanding_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "domain_understanding",
                "status": "success",
                "summary": "Domain understanding completed.",
                "artifacts": {
                    "domain_understanding_path": str(domain_understanding_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "domain_understanding",
                "status": "failed",
                "summary": "Domain understanding failed.",
                "reason": str(e),
            }],
        }

In [30]:
from pathlib import Path
import json


def dd_schema_detection_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    target_column = state.get("target_column")
    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})

    task = f"""
Stage: schema_detection

Read the training dataset from:
{dataset_path}

Target column from user:
{target_column}

Previous results:
basic_overview:
{json.dumps(basic, ensure_ascii=False, indent=2)}

domain_understanding:
{json.dumps(domain, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Inspect:
   - df.dtypes
   - df.nunique()
   - sample values for every object/string column
   - average string length for every object/string column
   - date-like string columns
   - id-like columns
   - boolean-like columns
3. Detect schema groups:
   - numeric
   - categorical
   - text
   - datetime
   - boolean_like
   - id
   - target

Strict rules:
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Do NOT classify numeric columns as datetime.
- Do NOT parse numeric columns as dates.
- Datetime detection applies only to object/string/datetime columns.
- Long free-form string columns should be text, not categorical.
- Columns like name, description, neighborhood_overview, host_about, amenities are text if they contain phrases, descriptions, lists, or natural language.
- ID-like columns such as id, host_id, user_id should be id, not numeric.
- Categorical columns are short string columns with repeated categories.
- If target_column does not exist in df, set target_column to null.
- If target_column is None, infer target only if obvious. Otherwise null.
- Every column should appear in at most one feature group, except target tracking.

Return JSON only:
{{
  "stage": "schema_detection",
  "status": "success",
  "result": {{
    "target_column": null,
    "task_type": "unknown",
    "schema": {{
      "numeric": [],
      "categorical": [],
      "text": [],
      "datetime": [],
      "boolean_like": [],
      "id": [],
      "target": []
    }},
    "schema_notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("schema_detection", task)
        schema_detection = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        schema_detection_path = artifacts_dir / "schema_detection.json"

        with open(schema_detection_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "schema_detection": schema_detection,
            "artifacts": {
                **state.get("artifacts", {}),
                "schema_detection_path": str(schema_detection_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "schema_detection",
                "status": "success",
                "summary": "Schema detection completed.",
                "artifacts": {
                    "schema_detection_path": str(schema_detection_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "schema_detection",
                "status": "failed",
                "summary": "Schema detection failed.",
                "reason": str(e),
            }],
        }

In [31]:
from pathlib import Path
import json


def dd_data_quality_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: data_quality

Read the training dataset from:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Calculate:
   - missing values by column
   - total missing values
   - duplicate rows
   - constant columns
   - high-cardinality columns
   - numeric outliers using IQR only for schema["numeric"]

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Use schema["numeric"] from the provided schema result.
- If schema["numeric"] is empty, return an empty numeric_outliers_iqr dictionary.
- Return only compact JSON for this stage.
- The values in final JSON must be copied exactly from variables calculated by python_interpreter_tool. Do not estimate or invent values.

Return JSON only:
{{
  "stage": "data_quality",
  "status": "success",
  "result": {{
    "missing_values": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0,
    "constant_columns": [],
    "high_cardinality_columns": [],
    "numeric_outliers_iqr": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("data_quality", task)
        data_quality = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        data_quality_path = artifacts_dir / "data_quality.json"

        with open(data_quality_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "data_quality": data_quality,
            "artifacts": {
                **state.get("artifacts", {}),
                "data_quality_path": str(data_quality_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "data_quality",
                "status": "success",
                "summary": "Data quality analysis completed.",
                "artifacts": {
                    "data_quality_path": str(data_quality_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "data_quality",
                "status": "failed",
                "summary": "Data quality analysis failed.",
                "reason": str(e),
            }],
        }

In [32]:
from pathlib import Path
import json


def dd_statistics_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: statistics

Read the training dataset from:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Use schema and target_column from the provided schema result.
3. Calculate detailed statistics:
   - numeric summary for schema["numeric"]
   - categorical summary for schema["categorical"]
   - text summary for schema["text"]
   - datetime summary for schema["datetime"]
   - target properties if target_column exists

Important rules:
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Do not return full detailed statistics in the final JSON.
- Return only compact metadata for this stage.

Return JSON only:
{{
  "stage": "statistics",
  "status": "success",
  "result": {{
    "numeric_columns_count": 0,
    "categorical_columns_count": 0,
    "text_columns_count": 0,
    "datetime_columns_count": 0,
    "target_column": null,
    "statistics_note": "Detailed statistics were calculated for EDA stage."
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("statistics", task)
        statistics = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        statistics_path = artifacts_dir / "statistics.json"

        with open(statistics_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "statistics": {
                **statistics,
                "statistics_path": str(statistics_path),
            },
            "artifacts": {
                **state.get("artifacts", {}),
                "statistics_path": str(statistics_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "statistics",
                "status": "success",
                "summary": "Statistics stage completed.",
                "artifacts": {
                    "statistics_path": str(statistics_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "statistics",
                "status": "failed",
                "summary": "Statistics calculation failed.",
                "reason": str(e),
            }],
        }

In [33]:
from pathlib import Path
import json


def dd_build_final_result_node(state: DataDescriptionState) -> DataDescriptionState:
    if state.get("error"):
        return {
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "skipped",
                "summary": "Final result build was skipped because previous stage failed.",
                "reason": state.get("error"),
            }],
        }

    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})
    schema_detection = state.get("schema_detection", {})
    data_quality = state.get("data_quality", {})
    statistics = state.get("statistics", {})
    previous_artifacts = state.get("artifacts", {})

    required_basic_keys = ["n_rows", "n_columns", "columns", "dtypes"]
    missing_basic_keys = [key for key in required_basic_keys if key not in basic]

    if missing_basic_keys:
        error = f"Cannot build final result. Missing basic_overview keys: {missing_basic_keys}"
        return {
            "error": error,
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "failed",
                "summary": "Final result build failed.",
                "reason": error,
            }],
        }

    schema = schema_detection.get("schema", {})

    for key in ["numeric", "categorical", "text", "datetime"]:
        schema.setdefault(key, [])

    for key in ["boolean_like", "id", "target"]:
        schema.setdefault(key, [])

    target_column = schema_detection.get("target_column")
    task_type = schema_detection.get("task_type", "unknown")

    if target_column is None:
        task_type = "unknown"

    artifacts_dir = Path("artifacts/data_description")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    summary_path = artifacts_dir / "data_description_summary.json"
    eda_artifacts_path = artifacts_dir / "eda_artifacts.json"
    report_path = artifacts_dir / "data_description_report.md"

    statistics_path = statistics.get(
        "statistics_path",
        previous_artifacts.get("statistics_path"),
    )

    artifacts = {
        **previous_artifacts,
        "data_description_summary_path": str(summary_path),
        "eda_artifacts_path": str(eda_artifacts_path),
        "data_description_report_path": str(report_path),
        "statistics_path": statistics_path,
        "eda_stages_dir": "artifacts/data_description/stages",
    }

    data_summary = {
        "n_rows": basic.get("n_rows"),
        "n_columns": basic.get("n_columns"),
        "domain_summary": domain.get("domain_summary"),
        "row_meaning": domain.get("row_meaning"),
        "missing_values_total": data_quality.get("missing_values_total"),
        "duplicate_rows": data_quality.get("duplicate_rows"),
        "statistics_path": statistics_path,
        "eda_artifacts_path": str(eda_artifacts_path),
    }

    final_result = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Dataset contains {basic.get('n_rows')} rows and "
            f"{basic.get('n_columns')} columns. Task type: {task_type}."
        ),
        "decisions": {
            "domain_summary": domain.get("domain_summary"),
            "row_meaning": domain.get("row_meaning"),
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_quality_summary": {
                "missing_values_total": data_quality.get("missing_values_total"),
                "duplicate_rows": data_quality.get("duplicate_rows"),
                "constant_columns_count": len(data_quality.get("constant_columns", [])),
                "high_cardinality_columns_count": len(data_quality.get("high_cardinality_columns", [])),
            },
        },
        "artifacts": artifacts,
        "next_input": {
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_summary": data_summary,
        },
        "reason": None,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)

    eda_artifacts = {
        "basic_overview": basic,
        "domain_understanding": domain,
        "schema_detection": schema_detection,
        "data_quality": data_quality,
        "statistics": statistics,
        "stage_artifact_paths": {
            key: value
            for key, value in artifacts.items()
            if key.endswith("_path") or key.endswith("_dir")
        },
    }

    with open(eda_artifacts_path, "w", encoding="utf-8") as f:
        json.dump(eda_artifacts, f, ensure_ascii=False, indent=2)

    report_md = (
        "# Data Description Report\n\n"
        "## Summary\n"
        f"{final_result.get('summary')}\n\n"
        "## Domain\n"
        f"{domain.get('domain_summary')}\n\n"
        "## Row meaning\n"
        f"{domain.get('row_meaning')}\n\n"
        "## Target\n"
        f"- Target column: {target_column}\n"
        f"- Task type: {task_type}\n\n"
        "## Dataset shape\n"
        f"- Rows: {basic.get('n_rows')}\n"
        f"- Columns: {basic.get('n_columns')}\n\n"
        "## Data quality summary\n"
        f"- Total missing values: {data_quality.get('missing_values_total')}\n"
        f"- Duplicate rows: {data_quality.get('duplicate_rows')}\n"
        f"- Constant columns count: {len(data_quality.get('constant_columns', []))}\n"
        f"- High-cardinality columns count: {len(data_quality.get('high_cardinality_columns', []))}\n\n"
        "## Schema\n"
        "```json\n"
        f"{json.dumps(schema, ensure_ascii=False, indent=2)}\n"
        "```\n\n"
        "## Artifacts\n"
        f"- Summary: {summary_path}\n"
        f"- EDA artifacts: {eda_artifacts_path}\n"
        f"- Statistics: {statistics_path}\n"
        f"- Stage artifacts dir: {artifacts.get('eda_stages_dir')}\n"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    print("\nRAW RESPONSE: build_final_result")
    print(json.dumps(final_result, ensure_ascii=False, indent=2))
    print("END RAW RESPONSE\n")

    return {
        "final_result": final_result,
        "artifacts": artifacts,
        "logs": [{
            "agent": "data_description_agent",
            "stage": "build_final_result",
            "status": "success",
            "summary": "Final compact data description result was built and saved.",
            "artifacts": {
                "data_description_summary_path": str(summary_path),
                "eda_artifacts_path": str(eda_artifacts_path),
                "data_description_report_path": str(report_path),
            },
        }],
    }

In [34]:
def route_after_dd_stage(state: DataDescriptionState) -> str:
    if state.get("error"):
        return "stop"
    return "continue"

In [36]:
from langgraph.graph import StateGraph, START, END


def build_data_description_graph():
    workflow = StateGraph(DataDescriptionState)

    workflow.add_node("basic_overview", dd_basic_overview_node)
    workflow.add_node("domain_understanding", dd_domain_understanding_node)
    workflow.add_node("schema_detection", dd_schema_detection_node)
    workflow.add_node("data_quality", dd_data_quality_node)
    workflow.add_node("statistics", dd_statistics_node)
    workflow.add_node("build_final_result", dd_build_final_result_node)

    workflow.add_edge(START, "basic_overview")

    workflow.add_conditional_edges(
        "basic_overview",
        route_after_dd_stage,
        {"continue": "domain_understanding", "stop": END},
    )

    workflow.add_conditional_edges(
        "domain_understanding",
        route_after_dd_stage,
        {"continue": "schema_detection", "stop": END},
    )

    workflow.add_conditional_edges(
        "schema_detection",
        route_after_dd_stage,
        {"continue": "data_quality", "stop": END},
    )

    workflow.add_conditional_edges(
        "data_quality",
        route_after_dd_stage,
        {"continue": "statistics", "stop": END},
    )

    workflow.add_conditional_edges(
        "statistics",
        route_after_dd_stage,
        {"continue": "build_final_result", "stop": END},
    )

    workflow.add_conditional_edges(
        "build_final_result",
        route_after_dd_stage,
        {"continue": END, "stop": END},
    )

    return workflow.compile()

In [37]:
def data_description_agent_node(state: AgentState) -> AgentState:
    dataset_path = state.get("dataset_path")
    target_column = state.get("target_column")

    if not dataset_path:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Dataset path is missing.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": "dataset_path is None",
        }

        return {
            "current_step": "data_description_agent",
            "error": "dataset_path is missing",
            "logs": [log_record],
        }

    try:
        data_description_app = build_data_description_graph()

        dd_state = data_description_app.invoke(
            {
                "dataset_path": dataset_path,
                "target_column": target_column,
                "artifacts_dir": "artifacts/data_description",
                "logs": [],
                "error": None,
                "artifacts": {},
            },
            {"recursion_limit": 20},
        )

        final_result = dd_state.get("final_result", {})

        if dd_state.get("error") or not final_result:
            error = dd_state.get("error") or "Data Description subgraph did not produce final_result."

            log_record = {
                "agent": "data_description_agent",
                "status": "failed",
                "skipped": False,
                "summary": "Data Description subgraph failed.",
                "decisions": {},
                "artifacts": dd_state.get("artifacts", {}),
                "next_input": {},
                "reason": error,
                "subgraph_logs": dd_state.get("logs", []),
            }

            return {
                "current_step": "data_description_agent",
                "error": error,
                "artifacts": {
                    **state.get("artifacts", {}),
                    **dd_state.get("artifacts", {}),
                },
                "logs": [log_record],
            }

        next_input = final_result.get("next_input", {})
        decisions = final_result.get("decisions", {})
        artifacts = final_result.get("artifacts", {})

        schema = next_input.get("schema") or decisions.get("schema", {})
        data_summary = next_input.get("data_summary", {})

        log_record = {
            "agent": "data_description_agent",
            "status": final_result.get("status", "success"),
            "skipped": final_result.get("skipped", False),
            "summary": final_result.get("summary", "Data description completed."),
            "decisions": decisions,
            "artifacts": artifacts,
            "next_input": next_input,
            "reason": final_result.get("reason"),
            "subgraph_logs": dd_state.get("logs", []),
        }

        return {
            "current_step": "data_description_agent",
            "target_column": next_input.get("target_column", target_column),
            "task_type": next_input.get("task_type", state.get("task_type")),
            "schema": schema,
            "data_summary": data_summary,
            "artifacts": {
                **state.get("artifacts", {}),
                **artifacts,
            },
            "logs": [log_record],
            "error": None,
        }

    except Exception as e:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Data Description subgraph failed.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e),
        }

        return {
            "current_step": "data_description_agent",
            "error": str(e),
            "logs": [log_record],
        }

In [36]:
from langgraph.graph import StateGraph, START, END


def build_start_test_graph():
    workflow = StateGraph(AgentState)

    # init-ноды
    workflow.add_node("dataset_finder", dataset_finder_node)
    workflow.add_node("previous_run_finder", previous_run_finder_node)
    workflow.add_node("join_init", join_init_node)

    # оркестратор
    workflow.add_node("orchestrator_agent", orchestrator_agent_node)

    # первый агент пайплайна
    workflow.add_node("data_description_agent", data_description_agent_node)

    # параллельный старт
    workflow.add_edge(START, "dataset_finder")
    workflow.add_edge(START, "previous_run_finder")

    # join после двух init-нод
    workflow.add_edge(["dataset_finder", "previous_run_finder"], "join_init")

    # после init идем в оркестратор
    workflow.add_edge("join_init", "orchestrator_agent")

    # проверяем только первый выбор оркестратора
    workflow.add_conditional_edges(
        "orchestrator_agent",
        route_by_orchestrator_decision,
        {
            "run_data_description": "data_description_agent",
            "run_tabular_preparation": END,
            "run_text_preparation": END,
            "merge_features": END,
            "run_modeling": END,
            "run_improvement": END,
            "run_reporting": END,
            "retry_current_step": END,
            "use_fallback": END,
            "finish": END,
            "fail": END,
        },
    )

    # после data_description останавливаем тест
    workflow.add_edge("data_description_agent", END)

    return workflow.compile()

In [37]:
from uuid import uuid4


def create_initial_state() -> AgentState:
    return {
        "run_id": str(uuid4()),
        "dataset_path": None,
        "target_column": "price",
        "task_type": "regression",

        "schema": {},
        "data_summary": {},
        "prep_summary": {},
        "text_summary": {
            "used": False,
            "text_columns": [],
        },
        "modeling_summary": {},

        "artifacts": {},
        "logs": [],

        "current_step": None,
        "next_action": None,
        "orchestration_decision": {},
        "orchestration_history": [],

        "previous_run": {},
        "previous_best_model": {},

        "retry_count": 0,
        "max_retries": 1,
        "status": "running",
        "error": None,
    }

In [ ]:
start_test_app = build_start_test_graph()

test_state = start_test_app.invoke(
    create_initial_state(),
    {"recursion_limit": 10}
)


========== START STAGE: basic_overview ==========

RAW RESPONSE:
{
  "stage": "basic_overview",
  "status": "success",
  "result": {
    "n_rows": 15000,
    "n_columns": 36,
    "columns": [
      "id",
      "description",
      "amenities",
      "property_type",
      "room_type",
      "accommodates",
      "bathrooms",
      "bedrooms",
      "beds",
      "host_since",
      "host_location",
      "host_response_time",
      "host_response_rate",
      "host_acceptance_rate",
      "host_is_superhost",
      "host_listings_count",
      "host_total_listings_count",
      "latitude",
      "longitude",
      "city",
      "has_availability",
      "availability_30",
      "availability_60",
      "availability_90",
      "availability_365",
      "number_of_reviews",
      "number_of_reviews_ltm",
      "number_of_reviews_l30d",
      "first_review",
      "last_review",
      "review_scores_rating",
      "review_scores_cleanliness",
      "review_scores_location",
      "revie

: 

In [ ]:
for i, log in enumerate(test_state.get("logs", []), 1):
    print(i, log["agent"], "-", log["status"], "-", log["summary"])

1 dataset_finder - success - Dataset found: ./rent_predictions/airbnb_train_fe_15000.csv
2 previous_run_finder - success - No previous run was found.
3 join_init - success - Initialization steps completed: dataset search and previous run search are done.
4 orchestrator_agent - success - The data_description_agent has not been completed as the schema and data_summary are currently empty and missing required keys such as n_rows, n_columns, and feature types.
5 data_description_agent - success - Dataset contains 15000 rows and 36 columns. Task type: regression.


: 

In [ ]:
print("dataset_path:", test_state.get("dataset_path"))
print("previous_run:", test_state.get("previous_run"))
print("previous_best_model:", test_state.get("previous_best_model"))
print("next_action:", test_state.get("next_action"))
print("current_step:", test_state.get("current_step"))
print("error:", test_state.get("error"))

dataset_path: ./rent_predictions/airbnb_train_fe_15000.csv
previous_run: {'exists': False, 'path': None}
previous_best_model: {}
next_action: run_data_description
current_step: data_description_agent
error: None


: 

In [ ]:
last_log = test_state.get("logs", [])[-1]

print("status:", last_log["status"])
print("summary:", last_log["summary"])
print("reason:", last_log["reason"])
print("error:", test_state.get("error"))

status: success
summary: Dataset contains 15000 rows and 36 columns. Task type: regression.
reason: None
error: None


: 

: 

## Делаем агента для обработки данных

In [38]:
from typing import TypedDict, Any


class DataPreparationState(TypedDict, total=False):
    # Base input from orchestrator
    dataset_path: str
    current_dataset_path: str

    # Input from Data Description Agent
    next_input: dict[str, Any]

    # Stage results
    cast_feature_types: dict[str, Any]
    feature_engineering: dict[str, Any]
    clean_columns: dict[str, Any]
    missing_values: dict[str, Any]
    encoding: dict[str, Any]
    final_preparation: dict[str, Any]

    # Final summary
    preparation_summary: dict[str, Any]
    created_features: list[dict[str, Any]]

    # Shared metadata
    artifacts: dict[str, Any]
    logs: list[dict[str, Any]]
    error: str
    status: str

In [39]:
def run_stage_with_llm_prep(stage_name: str, task: str) -> dict:
    print(f"\n========== START STAGE: {stage_name} ==========")

    try:
        result = data_preparation_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        print("\nGENERATED PYTHON CODE:")
        code_found = False

        for message in result["messages"]:
            tool_calls = getattr(message, "tool_calls", None)

            if not tool_calls and isinstance(message, dict):
                tool_calls = message.get("tool_calls")

            if tool_calls:
                for tool_call in tool_calls:
                    tool_name = tool_call.get("name")
                    args = tool_call.get("args", {})

                    if tool_name == "python_interpreter_tool":
                        code = args.get("code")

                        if code:
                            code_found = True
                            print("\n" + "-" * 60)
                            print(code)
                            print("-" * 60)

        if not code_found:
            print("No Python code tool call found.")

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise

In [40]:
from pathlib import Path
import json


def dp_cast_feature_types_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")
    schema = next_input.get("schema", {})

    output_path = "artifacts/data_preparation/intermediate/01_casted.csv"

    task = f"""
Stage: cast_feature_types

Read the training dataset from:
{dataset_path}

Target column:
{target_column}

Task type:
{task_type}

Schema from Data Description Agent:
{json.dumps(schema, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Create the output directory if it does not exist.
3. Convert columns according to the provided schema:
   - numeric columns: use pd.to_numeric(errors="coerce")
   - datetime columns: use pd.to_datetime(errors="coerce")
   - categorical columns: keep as object/string-like values, preserving missing values
   - text columns: keep as object/string-like values, preserving missing values
   - boolean_like columns: convert common boolean values to 1/0 where possible
   - target column: convert to numeric because the task type is regression
   - id columns: keep unchanged at this stage
4. Use only columns that actually exist in df.
5. Save the converted dataset to:
{output_path}

Important rules:
- Do not redefine the schema.
- Do not create new features in this stage.
- Do not remove duplicates in this stage.
- Do not handle missing values in this stage.
- Do not encode categorical columns.
- Do not scale numeric columns.
- Do not drop columns.
- Do not train models.
- Do not modify the original dataset file.
- Do not print Python code or debug output.
- Do not print the full dataframe.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "cast_feature_types",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "target_column": "{target_column}",
    "task_type": "{task_type}",
    "converted_columns": {{
      "numeric": [],
      "datetime": [],
      "categorical": [],
      "text": [],
      "boolean_like": [],
      "target": []
    }},
    "skipped_columns": [],
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("cast_feature_types", task)
        cast_feature_types = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        intermediate_dir = Path("artifacts/data_preparation/intermediate")
        intermediate_dir.mkdir(parents=True, exist_ok=True)

        cast_feature_types_path = artifacts_dir / "cast_feature_types.json"

        with open(cast_feature_types_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "cast_feature_types": cast_feature_types,
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "casted_dataset_path": output_path,
                "cast_feature_types_path": str(cast_feature_types_path),
            },
            "logs": [{
                "agent": "data_preparation_agent",
                "stage": "cast_feature_types",
                "status": "success",
                "summary": "Feature types casting completed.",
                "artifacts": {
                    "casted_dataset_path": output_path,
                    "cast_feature_types_path": str(cast_feature_types_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_preparation_agent",
                "stage": "cast_feature_types",
                "status": "failed",
                "summary": "Feature types casting failed.",
                "reason": str(e),
            }],
        }

In [41]:
data_prep_state = {
    "dataset_path": "./rent_predictions/airbnb_train_fe_15000.csv",
    "current_dataset_path": "./rent_predictions/airbnb_train_fe_15000.csv",
    "next_input": {
        "target_column": "price",
        "task_type": "regression",
        "schema": {
            "numeric": [
                "accommodates",
                "bathrooms",
                "bedrooms",
                "beds",
                "host_listings_count",
                "host_total_listings_count",
                "latitude",
                "longitude",
                "availability_30",
                "availability_60",
                "availability_90",
                "availability_365",
                "number_of_reviews",
                "number_of_reviews_ltm",
                "number_of_reviews_l30d",
                "review_scores_rating",
                "review_scores_cleanliness",
                "review_scores_location",
                "review_scores_value",
                "reviews_per_month"
            ],
            "categorical": [
                "property_type",
                "room_type",
                "host_location",
                "host_response_time",
                "host_response_rate",
                "host_acceptance_rate",
                "city"
            ],
            "text": [
                "description",
                "amenities"
            ],
            "datetime": [
                "host_since",
                "first_review",
                "last_review"
            ],
            "boolean_like": [
                "host_is_superhost",
                "has_availability"
            ],
            "id": [
                "id"
            ],
            "target": [
                "price"
            ]
        }
    },
    "cast_feature_types": {},
    "feature_engineering": {},
    "clean_columns": {},
    "missing_values": {},
    "encoding": {},
    "final_preparation": {},
    "preparation_summary": {},
    "created_features": [],
    "artifacts": {},
    "logs": [],
    "status": "running",
}

In [ ]:
state_after_cast = dp_cast_feature_types_node(data_prep_state)

print(state_after_cast.get("status"))
print(state_after_cast.get("error"))
print(state_after_cast.get("artifacts"))


========== START STAGE: cast_feature_types ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path

input_path = './rent_predictions/airbnb_train_fe_15000.csv'
output_path = 'artifacts/data_preparation/intermediate/01_casted.csv'
schema = {
  "numeric": ["accommodates", "bathrooms", "bedrooms", "beds", "host_listings_count", "host_total_listings_count", "latitude", "longitude", "availability_30", "availability_60", "availability_90", "availability_365", "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "review_scores_rating", "review_scores_cleanliness", "review_scores_location", "review_scores_value", "reviews_per_month"],
  "categorical": ["property_type", "room_type", "host_location", "host_response_time", "host_response_rate", "host_acceptance_rate", "city"],
  "text": ["description", "amenities"],
  "datetime": ["host_since", "first_review", "last_review"

: 

In [ ]:
data = pd.read_csv('artifacts/data_preparation/intermediate/01_casted.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         15000 non-null  int64  
 1   description                14644 non-null  str    
 2   amenities                  15000 non-null  str    
 3   property_type              15000 non-null  str    
 4   room_type                  15000 non-null  str    
 5   accommodates               15000 non-null  int64  
 6   bathrooms                  14995 non-null  float64
 7   bedrooms                   14986 non-null  float64
 8   beds                       14964 non-null  float64
 9   host_since                 14995 non-null  str    
 10  host_location              11113 non-null  str    
 11  host_response_time         12699 non-null  str    
 12  host_response_rate         12699 non-null  str    
 13  host_acceptance_rate       13718 non-null  str    
 14  h

: 

In [ ]:
data.head()

,id,description,amenities,property_type,room_type,accommodates,bathrooms,bedrooms,beds,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,latitude,longitude,city,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,reviews_per_month,price
0,206954,"Great spot in an excellent neighborhood, you c...","[""Self check-in"", ""Essentials"", ""Stove"", ""Cook...",Entire rental unit,Entire home/apt,1,1.0,1.0,1.0,2013-01-13,"San Francisco, CA",within an hour,100%,96%,1.0,5.0,7.0,37.802890,-122.419860,San Francisco,1.0,15,23,23,83,14,1,0,2013-08-01,2024-04-27,4.86,4.69,4.92,4.85,0.10,143.0
1,135896,This apartment was just renovated and it was t...,"[""Lockbox"", ""Bed linens"", ""Laundromat nearby"",...",Entire vacation home,Entire home/apt,5,1.0,1.0,3.0,2017-02-16,"Milan, Italy",within a few hours,100%,83%,NaN,16.0,24.0,45.451370,9.149250,Milan,1.0,23,43,64,229,12,4,1,2023-06-06,2025-03-02,4.92,4.92,4.75,4.75,0.56,71.0
2,30260,"Welcome to our stunning beachfront apartment, ...","[""Outdoor dining area"", ""Fire extinguisher"", ""...",Entire rental unit,Entire home/apt,8,2.0,2.0,4.0,2017-02-11,NaN,within an hour,100%,100%,0.0,2.0,2.0,25.987412,-80.119025,Broward County,1.0,24,37,61,135,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
3,82045,"Welcome to Studio De Luxe, your fully outfitte...","[""Elevator"", ""Dining table"", ""Refrigerator"", ""...",Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,2016-01-21,"Montpellier, France",within an hour,100%,100%,NaN,6.0,9.0,38.703430,-9.398130,Lisbon,1.0,29,59,89,364,56,17,0,2021-08-17,2024-09-26,4.89,4.96,4.88,4.70,1.28,126.0
4,26272,Modern bright and airy Studio. Well equipped w...,"[""Smoke alarm"", ""Dishes and silverware"", ""Exte...",Entire serviced apartment,Entire home/apt,2,1.0,0.0,1.0,2020-02-11,"England, United Kingdom",within an hour,100%,100%,1.0,1.0,2.0,51.465600,-2.617480,Bristol,1.0,15,32,51,132,408,79,5,2020-02-15,2024-12-17,4.95,4.94,4.95,4.88,6.89,94.0


: 

In [42]:
from pathlib import Path
import json


def dp_create_new_features_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["current_dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")
    schema = next_input.get("schema", {})

    previous_stage = state.get("cast_feature_types", {})

    output_path = "artifacts/data_preparation/intermediate/02_features.csv"

    task = f"""
Stage: create_new_features

Read the current dataset from:
{dataset_path}

Previous stage result:
{json.dumps(previous_stage, ensure_ascii=False, indent=2)}

Target column:
{target_column}

Task type:
{task_type}

Schema from Data Description Agent:
{json.dumps(schema, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

You are performing feature engineering for an Airbnb price prediction task.

Your goal:
Create meaningful new features that may help a regression model predict price.

Prompting strategy:
- Think step by step internally, but do not output your reasoning.
- Use the provided schema as the source of truth.
- First inspect only the column names and small samples if needed.
- Then choose useful feature transformations.
- Execute only this stage.
- Return only compact JSON.

Feature engineering rules:
1. Create at least 2 meaningful new features.
2. Prefer 4-8 useful features if the required source columns exist.
3. Use only columns that exist in df.
4. Do not use the target column to create features.
5. Do not create random or meaningless features.
6. Do not drop the target column.
7. Do not handle missing values globally in this stage.
8. Do not encode categorical columns in this stage.
9. Do not scale numeric columns in this stage.
10. Do not train models.

General feature engineering examples:
- From numeric columns:
  - ratio features: numeric_col_1 / numeric_col_2
  - difference features: numeric_col_1 - numeric_col_2
  - sum features: numeric_col_1 + numeric_col_2
  - product features: numeric_col_1 * numeric_col_2
  - squared features: numeric_col ** 2
  - log-transformed features: log1p(abs(numeric_col))
  - per-unit features, for example value per count, amount per duration, area per room

- From datetime columns:
  - year extracted from a date column
  - month extracted from a date column
  - day of week extracted from a date column
  - days since a date column
  - time difference between two date columns
  - age or duration features based on a start date and a reference date

- From text columns:
  - text length in characters
  - word count
  - average word length
  - count of specific separators or list-like elements
  - missing text indicator
  - simple keyword indicator if the keyword is clearly meaningful from sample values

- From categorical columns:
  - frequency/count encoding for high-cardinality categories
  - extracting a country, region, domain, or other subpart from structured strings if a clear separator exists
  - grouping rare categories into "other"
  - missing category indicator

- From percentage-like columns:
  - convert values like "85%" into numeric ratio values between 0 and 1
  - convert values like "85%" into numeric percentage values between 0 and 100

- From boolean-like columns:
  - normalize common boolean values into 1/0
  - create missing indicator if missingness may be informative

- From missing values:
  - missing indicator for columns with a meaningful missing share
  - count of missing values per row
  - share of missing values per row

- From grouped column families:
  - aggregate related numeric columns using mean, sum, max, min
  - create ratios between short-term and long-term measures if the column names suggest different time windows
  - create interaction features between capacity, activity, amount, or score-like columns

Feature selection rules:
- Create at least 2 meaningful new features if possible.
- Prefer features that are supported by column names, schema, and sample values.
- Do not create dataset-specific features unless the column names clearly support them.
- Do not use the target column to create features.
- Do not create random or meaningless features.
- Do not create ratios when the denominator can be zero without replacing zero with np.nan.
- If datetime columns are read from CSV as strings, convert them with pd.to_datetime(errors="coerce") before using them.
- If percentage-like values contain "%", remove "%" before numeric conversion.
- If a structured string contains separators such as comma, slash, pipe, semicolon, or brackets, you may extract simple count or last-part features when meaningful.

Your code must:
1. Read the CSV into a local variable df.
2. Create the output directory if it does not exist.
3. Create new features based on the available columns.
4. Store a Python list named created_features with dictionaries:
   {{
     "name": "...",
     "source_columns": [...],
     "reason": "..."
   }}
5. Store a Python list named skipped_features for features that could not be created.
6. Save the dataset with new features to:
{output_path}
7. Print only a compact JSON object with created_features and skipped_features.

Important rules:
- Do not print Python code.
- Do not print debug output.
- Do not print the full dataframe.
- Do not modify the original dataset file.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "create_new_features",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "target_column": "{target_column}",
    "created_features": [
      {{
        "name": "feature_name",
        "source_columns": [],
        "reason": "why this feature may help price prediction"
      }}
    ],
    "skipped_features": [],
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("create_new_features", task)
        feature_engineering = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        intermediate_dir = Path("artifacts/data_preparation/intermediate")
        intermediate_dir.mkdir(parents=True, exist_ok=True)

        feature_engineering_path = artifacts_dir / "create_new_features.json"

        with open(feature_engineering_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_preparation_agent",
            "stage": "create_new_features",
            "status": "success",
            "summary": "Feature engineering completed.",
            "artifacts": {
                "feature_dataset_path": output_path,
                "feature_engineering_path": str(feature_engineering_path),
            },
        }

        return {
            **state,
            "feature_engineering": feature_engineering,
            "created_features": feature_engineering.get("created_features", []),
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "feature_dataset_path": output_path,
                "feature_engineering_path": str(feature_engineering_path),
            },
            "logs": state.get("logs", []) + [new_log],
            "status": "running",
        }

    except Exception as e:
        new_log = {
            "agent": "data_preparation_agent",
            "stage": "create_new_features",
            "status": "failed",
            "summary": "Feature engineering failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

In [ ]:
state_after_features = dp_create_new_features_node(state_after_cast)

print("Status:", state_after_features.get("status"))
print("Error:", state_after_features.get("error"))
print("Current dataset path:", state_after_features.get("current_dataset_path"))
print("Artifacts:", state_after_features.get("artifacts"))


========== START STAGE: create_new_features ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
import json

input_path = 'artifacts/data_preparation/intermediate/01_casted.csv'
output_path = 'artifacts/data_preparation/intermediate/02_features.csv'

df = pd.read_csv(input_path)

# Ensure datetime columns are parsed
datetime_cols = ['host_since', 'first_review', 'last_review']
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

created_features = []
skipped_features = []

# 1. beds_per_bedroom
if 'beds' in df.columns and 'bedrooms' in df.columns:
    df['beds_per_bedroom'] = df['beds'] / df['bedrooms'].replace(0, np.nan)
    created_features.append({
        "name": "beds_per_bedroom",
        "source_columns": ["beds", "bedrooms"],
        "reason": "Measures density of sleeping arrangements which impacts pricing stra

: 

In [ ]:
data = pd.read_csv('artifacts/data_preparation/intermediate/02_features.csv')
data.head()

,id,description,amenities,property_type,room_type,accommodates,bathrooms,bedrooms,beds,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,latitude,longitude,city,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,reviews_per_month,price,beds_per_bedroom,amenities_count,host_tenure_days,description_length,reviews_per_listing_ratio,availability_30_ratio
0,206954,"Great spot in an excellent neighborhood, you c...","[""Self check-in"", ""Essentials"", ""Stove"", ""Cook...",Entire rental unit,Entire home/apt,1,1.0,1.0,1.0,2013-01-13,"San Francisco, CA",within an hour,100%,96%,1.0,5.0,7.0,37.802890,-122.419860,San Francisco,1.0,15,23,23,83,14,1,0,2013-08-01,2024-04-27,4.86,4.69,4.92,4.85,0.10,143.0,1.0,19,4436.0,432,2.800000,0.500000
1,135896,This apartment was just renovated and it was t...,"[""Lockbox"", ""Bed linens"", ""Laundromat nearby"",...",Entire vacation home,Entire home/apt,5,1.0,1.0,3.0,2017-02-16,"Milan, Italy",within a few hours,100%,83%,NaN,16.0,24.0,45.451370,9.149250,Milan,1.0,23,43,64,229,12,4,1,2023-06-06,2025-03-02,4.92,4.92,4.75,4.75,0.56,71.0,3.0,49,2941.0,536,0.750000,0.766667
2,30260,"Welcome to our stunning beachfront apartment, ...","[""Outdoor dining area"", ""Fire extinguisher"", ""...",Entire rental unit,Entire home/apt,8,2.0,2.0,4.0,2017-02-11,NaN,within an hour,100%,100%,0.0,2.0,2.0,25.987412,-80.119025,Broward County,1.0,24,37,61,135,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,2.0,18,2946.0,430,0.000000,0.800000
3,82045,"Welcome to Studio De Luxe, your fully outfitte...","[""Elevator"", ""Dining table"", ""Refrigerator"", ""...",Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,2016-01-21,"Montpellier, France",within an hour,100%,100%,NaN,6.0,9.0,38.703430,-9.398130,Lisbon,1.0,29,59,89,364,56,17,0,2021-08-17,2024-09-26,4.89,4.96,4.88,4.70,1.28,126.0,1.0,46,3333.0,499,9.333333,0.966667
4,26272,Modern bright and airy Studio. Well equipped w...,"[""Smoke alarm"", ""Dishes and silverware"", ""Exte...",Entire serviced apartment,Entire home/apt,2,1.0,0.0,1.0,2020-02-11,"England, United Kingdom",within an hour,100%,100%,1.0,1.0,2.0,51.465600,-2.617480,Bristol,1.0,15,32,51,132,408,79,5,2020-02-15,2024-12-17,4.95,4.94,4.95,4.88,6.89,94.0,NaN,34,1851.0,217,408.000000,0.500000


: 

In [43]:
from pathlib import Path
import json


def dp_clean_columns_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["current_dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")
    schema = next_input.get("schema", {})

    previous_stage = state.get("feature_engineering", {})
    created_features = state.get("created_features", [])

    output_path = "artifacts/data_preparation/intermediate/03_cleaned.csv"

    task = f"""
Stage: clean_columns

Read the current dataset from:
{dataset_path}

Previous stage result:
{json.dumps(previous_stage, ensure_ascii=False, indent=2)}

Created features:
{json.dumps(created_features, ensure_ascii=False, indent=2)}

Target column:
{target_column}

Task type:
{task_type}

Schema from Data Description Agent:
{json.dumps(schema, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

You are cleaning columns before missing value handling and encoding.

Your goal:
Remove columns that should not be used directly by a machine learning model after feature engineering.

Column cleaning rules:
1. Drop id columns from schema["id"] if they exist.
2. Drop raw datetime columns from schema["datetime"] if date-derived features were created or if they are still raw datetime/string columns.
3. Drop raw text columns from schema["text"] if text-derived features were created or if they are still long raw text fields.
4. Do not drop the target column.
5. Do not drop newly created numeric features.
6. Do not handle missing values globally in this stage.
7. Do not encode categorical columns.
8. Do not scale numeric columns.
9. Do not train models.

High-cardinality categorical rule:
- You may drop categorical columns with very high cardinality if they are unlikely to generalize.
- Use a conservative threshold: more than 100 unique values.
- Do not drop low-cardinality categorical columns.

Your code must:
1. Read the CSV into a local variable df.
2. Create the output directory if it does not exist.
3. Drop only columns that satisfy the rules above.
4. Store a Python list named dropped_columns with dictionaries:
   {{
     "name": "...",
     "reason": "..."
   }}
5. Store a Python list named kept_columns_notes if useful.
6. Save the cleaned dataset to:
{output_path}
7. Print only a compact JSON object with dropped_columns and warnings.

Important rules:
- Use only columns that actually exist in df.
- Do not print Python code.
- Do not print debug output.
- Do not print the full dataframe.
- Do not modify the original dataset file.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "clean_columns",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "target_column": "{target_column}",
    "dropped_columns": [
      {{
        "name": "column_name",
        "reason": "why this column was dropped"
      }}
    ],
    "kept_columns_notes": [],
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("clean_columns", task)
        clean_columns = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        intermediate_dir = Path("artifacts/data_preparation/intermediate")
        intermediate_dir.mkdir(parents=True, exist_ok=True)

        clean_columns_path = artifacts_dir / "clean_columns.json"

        with open(clean_columns_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_preparation_agent",
            "stage": "clean_columns",
            "status": "success",
            "summary": "Column cleaning completed.",
            "artifacts": {
                "cleaned_dataset_path": output_path,
                "clean_columns_path": str(clean_columns_path),
            },
        }

        return {
            **state,
            "clean_columns": clean_columns,
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "cleaned_dataset_path": output_path,
                "clean_columns_path": str(clean_columns_path),
            },
            "logs": state.get("logs", []) + [new_log],
            "status": "running",
        }

    except Exception as e:
        new_log = {
            "agent": "data_preparation_agent",
            "stage": "clean_columns",
            "status": "failed",
            "summary": "Column cleaning failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

In [ ]:
state_after_clean = dp_clean_columns_node(state_after_features)

print("Status:", state_after_clean.get("status"))
print("Error:", state_after_clean.get("error"))
print("Current dataset path:", state_after_clean.get("current_dataset_path"))
print("Artifacts:", state_after_clean.get("artifacts"))


========== START STAGE: clean_columns ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import os
import json
from pathlib import Path

input_path = 'artifacts/data_preparation/intermediate/02_features.csv'
output_path = 'artifacts/data_preparation/intermediate/03_cleaned.csv'

df = pd.read_csv(input_path)
dropped_columns = []
warnings = []

# 1. Drop ID columns (common patterns)
id_cols = [col for col in df.columns if col.lower() in ['id', 'listing_id', 'host_id']]
for col in id_cols:
    dropped_columns.append({"name": col, "reason": "ID column unlikely to generalize."})

# 2. Drop raw datetime columns (host_since was used for host_tenure_days)
if 'host_since' in df.columns:
    dropped_columns.append({"name": 'host_since', "reason": "Raw datetime column replaced by host_tenure_days."})

# 3. Drop raw text columns (description and amenities were used for features)
text_cols = ['description', 'amenities', 'name', 'sum

: 

In [ ]:
data = pd.read_csv('artifacts/data_preparation/intermediate/03_cleaned.csv')
data.head()

,property_type,room_type,accommodates,bathrooms,bedrooms,beds,host_response_time,host_response_rate,host_is_superhost,host_listings_count,host_total_listings_count,latitude,longitude,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,reviews_per_month,price,beds_per_bedroom,amenities_count,host_tenure_days,description_length,reviews_per_listing_ratio,availability_30_ratio
0,Entire rental unit,Entire home/apt,1,1.0,1.0,1.0,within an hour,100%,1.0,5.0,7.0,37.802890,-122.419860,1.0,15,23,23,83,14,1,0,4.86,4.69,4.92,4.85,0.10,143.0,1.0,19,4436.0,432,2.800000,0.500000
1,Entire vacation home,Entire home/apt,5,1.0,1.0,3.0,within a few hours,100%,NaN,16.0,24.0,45.451370,9.149250,1.0,23,43,64,229,12,4,1,4.92,4.92,4.75,4.75,0.56,71.0,3.0,49,2941.0,536,0.750000,0.766667
2,Entire rental unit,Entire home/apt,8,2.0,2.0,4.0,within an hour,100%,0.0,2.0,2.0,25.987412,-80.119025,1.0,24,37,61,135,0,0,0,NaN,NaN,NaN,NaN,NaN,500.0,2.0,18,2946.0,430,0.000000,0.800000
3,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,within an hour,100%,NaN,6.0,9.0,38.703430,-9.398130,1.0,29,59,89,364,56,17,0,4.89,4.96,4.88,4.70,1.28,126.0,1.0,46,3333.0,499,9.333333,0.966667
4,Entire serviced apartment,Entire home/apt,2,1.0,0.0,1.0,within an hour,100%,1.0,1.0,2.0,51.465600,-2.617480,1.0,15,32,51,132,408,79,5,4.95,4.94,4.95,4.88,6.89,94.0,NaN,34,1851.0,217,408.000000,0.500000


: 

In [44]:
from pathlib import Path
import json


def dp_handle_missing_values_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["current_dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")
    schema = next_input.get("schema", {})

    previous_stage = state.get("clean_columns", {})

    output_path = "artifacts/data_preparation/intermediate/04_no_missing.csv"

    task = f"""
Stage: handle_missing_values

Read the current dataset from:
{dataset_path}

Previous stage result:
{json.dumps(previous_stage, ensure_ascii=False, indent=2)}

Target column:
{target_column}

Task type:
{task_type}

Schema from Data Description Agent:
{json.dumps(schema, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

You are handling missing values before categorical encoding.

Your goal:
Create a dataset where missing values are handled in a simple, reproducible way.

Missing value handling rules:
1. Drop rows where the target column is missing.
2. For numeric columns, fill missing values with the median.
3. For boolean-like columns, fill missing values with the most frequent value if possible, otherwise 0.
4. For categorical columns, fill missing values with "missing".
5. For text-derived numeric features, fill missing values with the median.
6. For any remaining object/string columns, fill missing values with "missing".
7. For any remaining numeric columns, fill missing values with the median.
8. Do not encode categorical columns in this stage.
9. Do not scale numeric columns.
10. Do not train models.
11. Do not drop feature columns only because they contain missing values.

Important:
- Use only columns that actually exist in df.
- Do not modify the original dataset file.
- Do not use the target column for feature engineering.
- Do not print Python code or debug output.
- Do not print the full dataframe.
- Return only compact JSON for this stage.

Your code must:
1. Read the CSV into a local variable df.
2. Create the output directory if it does not exist.
3. Drop rows with missing target values.
4. Fill missing values according to the rules above.
5. Save the dataset to:
{output_path}
6. Store a Python list named missing_actions with dictionaries:
   {{
     "column": "...",
     "strategy": "...",
     "missing_before": 0,
     "missing_after": 0
   }}
7. Print only a compact JSON object with missing_actions and warnings.

Return JSON only:
{{
  "stage": "handle_missing_values",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "target_column": "{target_column}",
    "rows_before": null,
    "rows_after": null,
    "missing_actions": [
      {{
        "column": "column_name",
        "strategy": "median_or_missing_or_mode",
        "missing_before": 0,
        "missing_after": 0
      }}
    ],
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("handle_missing_values", task)
        missing_values = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        intermediate_dir = Path("artifacts/data_preparation/intermediate")
        intermediate_dir.mkdir(parents=True, exist_ok=True)

        missing_values_path = artifacts_dir / "handle_missing_values.json"

        with open(missing_values_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_preparation_agent",
            "stage": "handle_missing_values",
            "status": "success",
            "summary": "Missing values handling completed.",
            "artifacts": {
                "no_missing_dataset_path": output_path,
                "missing_values_path": str(missing_values_path),
            },
        }

        return {
            **state,
            "missing_values": missing_values,
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "no_missing_dataset_path": output_path,
                "missing_values_path": str(missing_values_path),
            },
            "logs": state.get("logs", []) + [new_log],
            "status": "running",
        }

    except Exception as e:
        new_log = {
            "agent": "data_preparation_agent",
            "stage": "handle_missing_values",
            "status": "failed",
            "summary": "Missing values handling failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

In [ ]:
state_after_missing = dp_handle_missing_values_node(state_after_clean)

print("Status:", state_after_missing.get("status"))
print("Error:", state_after_missing.get("error"))
print("Current dataset path:", state_after_missing.get("current_dataset_path"))
print("Artifacts:", state_after_missing.get("artifacts"))


========== START STAGE: handle_missing_values ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
import json

input_path = 'artifacts/data_preparation/intermediate/03_cleaned.csv'
output_path = 'artifacts/data_preparation/intermediate/04_no_missing.csv'
target_column = 'price'

df = pd.read_csv(input_path)
rows_before = len(df)

# 1. Drop rows where target is missing
if target_column in df.columns:
    df = df.dropna(subset=[target_column])
rows_after = len(df)

missing_actions = []
warnings = []

# Identify columns with missing values
cols_with_missing = df.columns[df.isnull().any()].tolist()

for col in df.columns:
    missing_before = int(df[col].isnull().sum())
    if missing_before == 0:
        continue
        
    strategy = ""
    if df[col].dtype == 'object' or df[col].dtype == 'bool':
        # Rule 3 & 4 & 6: Categorical/Boolean/Object
        if df[col].dtype == '

: 

In [ ]:
data = pd.read_csv('artifacts/data_preparation/intermediate/04_no_missing.csv')
data.head()

,property_type,room_type,accommodates,bathrooms,bedrooms,beds,host_response_time,host_response_rate,host_is_superhost,host_listings_count,host_total_listings_count,latitude,longitude,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,reviews_per_month,price,beds_per_bedroom,amenities_count,host_tenure_days,description_length,reviews_per_listing_ratio,availability_30_ratio
0,Entire rental unit,Entire home/apt,1,1.0,1.0,1.0,within an hour,100%,1.0,5.0,7.0,37.802890,-122.419860,1.0,15,23,23,83,14,1,0,4.86,4.69,4.92,4.85,0.10,143.0,1.0,19,4436.0,432,2.800000,0.500000
1,Entire vacation home,Entire home/apt,5,1.0,1.0,3.0,within a few hours,100%,0.0,16.0,24.0,45.451370,9.149250,1.0,23,43,64,229,12,4,1,4.92,4.92,4.75,4.75,0.56,71.0,3.0,49,2941.0,536,0.750000,0.766667
2,Entire rental unit,Entire home/apt,8,2.0,2.0,4.0,within an hour,100%,0.0,2.0,2.0,25.987412,-80.119025,1.0,24,37,61,135,0,0,0,4.86,4.87,4.88,4.77,0.87,500.0,2.0,18,2946.0,430,0.000000,0.800000
3,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,within an hour,100%,0.0,6.0,9.0,38.703430,-9.398130,1.0,29,59,89,364,56,17,0,4.89,4.96,4.88,4.70,1.28,126.0,1.0,46,3333.0,499,9.333333,0.966667
4,Entire serviced apartment,Entire home/apt,2,1.0,0.0,1.0,within an hour,100%,1.0,1.0,2.0,51.465600,-2.617480,1.0,15,32,51,132,408,79,5,4.95,4.94,4.95,4.88,6.89,94.0,1.0,34,1851.0,217,408.000000,0.500000


: 

In [45]:
from pathlib import Path
import json


def dp_encode_and_scale_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["current_dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")
    schema = next_input.get("schema", {})

    previous_stage = state.get("missing_values", {})

    output_path = "artifacts/data_preparation/intermediate/05_encoded_scaled.csv"

    task = f"""
Stage: encode_and_scale

Read the current dataset from:
{dataset_path}

Previous stage result:
{json.dumps(previous_stage, ensure_ascii=False, indent=2)}

Target column:
{target_column}

Task type:
{task_type}

Schema from Data Description Agent:
{json.dumps(schema, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

You are encoding categorical features and scaling numeric features before final ML dataset validation.

Your goal:
Convert all remaining FEATURE columns into numeric format and scale numeric FEATURE columns.
NEVER transform, encode, or scale the PRICE COLUMN
Critical target rules:
- The target column is not a feature.
- Never transform the target column.
- Never scale the target column.
- Never encode the target column.
- Never overwrite the target column with transformed values.
- Never include the target column in numeric_features.
- Never include the target column in categorical_features.
- Never pass the target column to get_dummies.
- Never include the target column in frequency encoding.
- The target column in the output file must remain in the original scale and unchanged.
- Store target_values = df[target_column].copy() before any transformations.
- Restore df[target_column] = target_values before saving the output dataset.

Encoding rules:
1. Detect all remaining object/string/category FEATURE columns in the current dataset.
2. Exclude the target column before detecting categorical columns.
3. Use pandas get_dummies for low-cardinality categorical feature columns.
4. Use drop_first=False to avoid losing category information.
5. Use dummy_na=False because missing values were already handled in the previous stage.
6. If a categorical feature column has more than 100 unique values, do not one-hot encode it.
7. For high-cardinality categorical feature columns, use frequency encoding:
   - create a new column named original_column + "_freq"
   - value = frequency of the category in the dataset
   - drop the original high-cardinality categorical column.

Scaling rules:
1. Scale feature columns only.
2. Exclude the target column before selecting numeric columns for scaling.
3. Build numeric_features exactly like this:
   numeric_features = [
       col for col in df.select_dtypes(include=["number"]).columns
       if col != target_column
   ]
4. Do not scale binary dummy columns with only 0/1 values.
5. Do not scale the target column under any circumstances.
6. Use standard scaling only for non-binary numeric feature columns:
   scaled_value = (value - mean) / standard_deviation
7. If a numeric feature column has zero standard deviation, keep it unchanged and add a warning.
8. Do not train models.
9. Do not remove rows.
10. Do not modify the original dataset file.

Your code must:
1. Read the CSV into a local variable df.
2. Store the target column separately before transformations:
   target_values = df[target_column].copy()
3. Create the output directory if it does not exist.
4. Identify remaining non-numeric feature columns, excluding the target column.
5. Apply one-hot encoding or frequency encoding only to feature columns.
6. After encoding, identify numeric feature columns with:
   numeric_features = [
       col for col in df.select_dtypes(include=["number"]).columns
       if col != target_column
   ]
7. Exclude binary dummy columns from scaling:
   binary_columns = columns where non-missing unique values are subset of {{0, 1}}
8. Scale only non-binary numeric feature columns.
9. Restore the target column unchanged before saving:
   df[target_column] = target_values
10. Save the encoded and scaled dataset to:
{output_path}
11. Store a Python list named encoding_actions with dictionaries:
   {{
     "column": "...",
     "strategy": "one_hot_or_frequency",
     "n_unique": 0,
     "created_columns_count": 0
   }}
12. Store a Python list named scaling_actions with dictionaries:
   {{
     "column": "...",
     "strategy": "standard_scaling",
     "mean": 0.0,
     "std": 1.0
   }}
13. Print only a compact JSON object with encoding_actions, scaling_actions and warnings.

Important rules:
- Use only columns that actually exist in df.
- Do not print Python code.
- Do not print debug output.
- Do not print the full dataframe.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "encode_and_scale",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "target_column": "{target_column}",
    "target_scaled": false,
    "target_encoded": false,
    "target_unchanged": true,
    "encoding_actions": [
      {{
        "column": "column_name",
        "strategy": "one_hot_or_frequency",
        "n_unique": 0,
        "created_columns_count": 0
      }}
    ],
    "scaling_actions": [
      {{
        "column": "column_name",
        "strategy": "standard_scaling",
        "mean": 0.0,
        "std": 1.0
      }}
    ],
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("encode_and_scale", task)
        encoding_scaling = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        intermediate_dir = Path("artifacts/data_preparation/intermediate")
        intermediate_dir.mkdir(parents=True, exist_ok=True)

        encoding_scaling_path = artifacts_dir / "encode_and_scale.json"

        with open(encoding_scaling_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "data_preparation_agent",
            "stage": "encode_and_scale",
            "status": "success",
            "summary": "Categorical encoding and numeric feature scaling completed. Target column was instructed to remain unchanged.",
            "artifacts": {
                "encoded_scaled_dataset_path": output_path,
                "encoding_scaling_path": str(encoding_scaling_path),
            },
        }

        return {
            **state,
            "encoding": encoding_scaling,
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "encoded_scaled_dataset_path": output_path,
                "encoding_scaling_path": str(encoding_scaling_path),
            },
            "logs": state.get("logs", []) + [new_log],
            "status": "running",
        }

    except Exception as e:
        new_log = {
            "agent": "data_preparation_agent",
            "stage": "encode_and_scale",
            "status": "failed",
            "summary": "Categorical encoding and numeric scaling failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

In [ ]:
state_after_encoding_scaling = dp_encode_and_scale_node(state_after_missing)

print("Status:", state_after_encoding_scaling.get("status"))
print("Error:", state_after_encoding_scaling.get("error"))
print("Current dataset path:", state_after_encoding_scaling.get("current_dataset_path"))
print("Artifacts:", state_after_encoding_scaling.get("artifacts"))


========== START STAGE: encode_and_scale ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
import json

input_path = 'artifacts/data_preparation/intermediate/04_no_missing.csv'
output_path = 'artifacts/data_preparation/intermediate/05_encoded_scaled.csv'
target_column = 'price'

df = pd.read_csv(input_path)
encoding_actions = []
scaling_actions = []
warnings = []

# 1. Encoding
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if target_column in categorical_cols:
    categorical_cols.remove(target_column)

for col in categorical_cols:
    n_unique = df[col].nunique()
    if n_unique > 100:
        # Frequency Encoding
        freq = df[col].value_counts(normalize=True)
        df[f'{col}_freq'] = df[col].map(freq)
        df.drop(columns=[col], inplace=True)
        encoding_actions.append({
            "column": col,
            "strategy"

: 

In [ ]:
data = pd.read_csv('artifacts/data_preparation/intermediate/05_encoded_scaled.csv')
data.head()

,accommodates,bathrooms,bedrooms,beds,host_is_superhost,host_listings_count,host_total_listings_count,latitude,longitude,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,reviews_per_month,price,beds_per_bedroom,amenities_count,host_tenure_days,description_length,reviews_per_listing_ratio,availability_30_ratio,property_type_Barn,property_type_Boat,property_type_Camper/RV,property_type_Campsite,property_type_Casa particular,property_type_Cycladic home,property_type_Dammuso,property_type_Dome,property_type_Earthen home,property_type_Entire bed and breakfast,property_type_Entire bungalow,property_type_Entire cabin,property_type_Entire chalet,property_type_Entire condo,property_type_Entire cottage,property_type_Entire guest suite,property_type_Entire guesthouse,property_type_Entire home,property_type_Entire home/apt,property_type_Entire loft,property_type_Entire place,...,host_response_rate_54%,host_response_rate_55%,host_response_rate_56%,host_response_rate_57%,host_response_rate_58%,host_response_rate_59%,host_response_rate_6%,host_response_rate_60%,host_response_rate_61%,host_response_rate_62%,host_response_rate_63%,host_response_rate_64%,host_response_rate_65%,host_response_rate_67%,host_response_rate_68%,host_response_rate_69%,host_response_rate_7%,host_response_rate_70%,host_response_rate_71%,host_response_rate_72%,host_response_rate_73%,host_response_rate_74%,host_response_rate_75%,host_response_rate_76%,host_response_rate_77%,host_response_rate_78%,host_response_rate_79%,host_response_rate_8%,host_response_rate_80%,host_response_rate_81%,host_response_rate_82%,host_response_rate_83%,host_response_rate_84%,host_response_rate_85%,host_response_rate_86%,host_response_rate_87%,host_response_rate_88%,host_response_rate_89%,host_response_rate_9%,host_response_rate_90%,host_response_rate_91%,host_response_rate_92%,host_response_rate_93%,host_response_rate_94%,host_response_rate_95%,host_response_rate_96%,host_response_rate_97%,host_response_rate_98%,host_response_rate_99%,host_response_rate_missing
0,-1.201366,-0.503651,-0.572045,-0.748127,1.0,-0.168191,-0.180711,0.214309,-1.527426,1.0,0.024393,-0.450043,-0.919746,-1.080196,-0.336301,-0.554650,-0.452571,0.253220,-0.222677,0.406745,0.432375,-0.813458,143.0,-0.509846,-0.886993,1.440810,0.422018,-0.324359,0.024393,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,0.441528,-0.503651,-0.572045,0.391153,0.0,-0.141911,-0.150115,0.523365,0.311177,1.0,0.731771,0.450854,0.338009,0.145648,-0.360089,-0.387041,0.224795,0.427246,0.426445,-0.165140,0.166324,-0.509446,71.0,2.295641,0.913107,0.320425,1.081527,-0.359359,0.731771,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,1.673698,0.776058,0.324952,0.960793,0.0,-0.175358,-0.189710,-0.263125,-0.936296,1.0,0.820194,0.180585,0.245978,-0.643594,-0.502814,-0.610520,-0.452571,0.253220,0.285332,0.272184,0.219534,-0.304568,500.0,0.892897,-0.946997,0.324173,0.409335,-0.372163,0.820194,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,F

: 

In [46]:
from pathlib import Path
import json


def dp_final_preparation_node(state: DataPreparationState) -> DataPreparationState:
    dataset_path = state["current_dataset_path"]
    next_input = state.get("next_input", {})

    target_column = next_input.get("target_column")
    task_type = next_input.get("task_type")

    output_path = "artifacts/data_preparation/prepared_dataset.csv"
    summary_path = "artifacts/data_preparation/data_preparation_summary.json"

    task = f"""
Stage: final_preparation

Read the current dataset from:
{dataset_path}

Target column:
{target_column}

Task type:
{task_type}

Use python_interpreter_tool to write and execute short Python code.

Your goal:
Validate and save the final ML-ready dataset.

Your code must:
1. Read the CSV into a local variable df.
2. Create the output directory if it does not exist.
3. Check that the target column exists.
4. Check that all feature columns are numeric.
5. Check missing values in the final dataset.
6. Save the final dataset to:
{output_path}
7. Save a compact JSON summary to:
{summary_path}

Validation rules:
- Do not train models.
- Do not modify the original dataset file.
- Do not drop the target column.
- Do not print Python code.
- Do not print debug output.
- Do not print the full dataframe.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "final_preparation",
  "status": "success",
  "result": {{
    "input_path": "{dataset_path}",
    "output_path": "{output_path}",
    "summary_path": "{summary_path}",
    "target_column": "{target_column}",
    "task_type": "{task_type}",
    "final_shape": null,
    "n_features": null,
    "non_numeric_feature_columns": [],
    "missing_values_total": null,
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_prep("final_preparation", task)
        final_preparation = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_preparation/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        final_preparation_path = artifacts_dir / "final_preparation.json"

        with open(final_preparation_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        preparation_summary = {
            "target_column": target_column,
            "task_type": task_type,
            "final_dataset_path": output_path,
            "summary_path": summary_path,
            "stage_artifacts": {
                "cast_feature_types": state.get("artifacts", {}).get("cast_feature_types_path"),
                "create_new_features": state.get("artifacts", {}).get("feature_engineering_path"),
                "clean_columns": state.get("artifacts", {}).get("clean_columns_path"),
                "handle_missing_values": state.get("artifacts", {}).get("missing_values_path"),
                "encode_and_scale": state.get("artifacts", {}).get("encoding_scaling_path"),
                "final_preparation": str(final_preparation_path),
            },
            "created_features": state.get("created_features", []),
            "final_preparation": final_preparation,
        }

        new_log = {
            "agent": "data_preparation_agent",
            "stage": "final_preparation",
            "status": "success",
            "summary": "Final ML-ready dataset saved.",
            "artifacts": {
                "prepared_dataset_path": output_path,
                "data_preparation_summary_path": summary_path,
                "final_preparation_path": str(final_preparation_path),
            },
        }

        return {
            **state,
            "final_preparation": final_preparation,
            "preparation_summary": preparation_summary,
            "current_dataset_path": output_path,
            "artifacts": {
                **state.get("artifacts", {}),
                "prepared_dataset_path": output_path,
                "data_preparation_summary_path": summary_path,
                "final_preparation_path": str(final_preparation_path),
            },
            "logs": state.get("logs", []) + [new_log],
            "status": "success",
        }

    except Exception as e:
        new_log = {
            "agent": "data_preparation_agent",
            "stage": "final_preparation",
            "status": "failed",
            "summary": "Final preparation failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

In [47]:
from langgraph.graph import StateGraph, START, END

data_preparation_graph = StateGraph(DataPreparationState)

data_preparation_graph.add_node("cast_feature_types", dp_cast_feature_types_node)
data_preparation_graph.add_node("create_new_features", dp_create_new_features_node)
data_preparation_graph.add_node("clean_columns", dp_clean_columns_node)
data_preparation_graph.add_node("handle_missing_values", dp_handle_missing_values_node)
data_preparation_graph.add_node("encode_and_scale", dp_encode_and_scale_node)
data_preparation_graph.add_node("final_preparation", dp_final_preparation_node)

data_preparation_graph.add_edge(START, "cast_feature_types")
data_preparation_graph.add_edge("cast_feature_types", "create_new_features")
data_preparation_graph.add_edge("create_new_features", "clean_columns")
data_preparation_graph.add_edge("clean_columns", "handle_missing_values")
data_preparation_graph.add_edge("handle_missing_values", "encode_and_scale")
data_preparation_graph.add_edge("encode_and_scale", "final_preparation")
data_preparation_graph.add_edge("final_preparation", END)

data_preparation_app = data_preparation_graph.compile()

In [ ]:
final_data_prep_state = data_preparation_app.invoke(data_prep_state)

print(final_data_prep_state["status"])
print(final_data_prep_state.get("error"))
print(final_data_prep_state["artifacts"]["prepared_dataset_path"])


========== START STAGE: cast_feature_types ==========

GENERATED PYTHON CODE:

------------------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path

input_path = './rent_predictions/airbnb_train_fe_15000.csv'
output_path = 'artifacts/data_preparation/intermediate/01_casted.csv'
schema = {
  "numeric": ["accommodates", "bathrooms", "bedrooms", "beds", "host_listings_count", "host_total_listings_count", "latitude", "longitude", "availability_30", "availability_60", "availability_90", "availability_365", "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "review_scores_rating", "review_scores_cleanliness", "review_scores_location", "review_scores_value", "reviews_per_month"],
  "categorical": ["property_type", "room_type", "host_location", "host_response_time", "host_response_rate", "host_acceptance_rate", "city"],
  "text": ["description", "amenities"],
  "datetime": ["host_since", "first_review", "last_review"

: 

In [48]:
def run_data_preparation_agent_node(state: AgentState) -> AgentState:
    print("=== RUN DATA PREPARATION AGENT ===")

    dataset_path = state["dataset_path"]

    next_input = state.get("next_input", {})

    data_prep_state: DataPreparationState = {
        "dataset_path": dataset_path,
        "current_dataset_path": dataset_path,
        "next_input": next_input,
        "cast_feature_types": {},
        "feature_engineering": {},
        "clean_columns": {},
        "missing_values": {},
        "encoding": {},
        "final_preparation": {},
        "preparation_summary": {},
        "created_features": [],
        "artifacts": {},
        "logs": [],
        "status": "running",
    }

    final_data_prep_state = data_preparation_app.invoke(
        data_prep_state,
        {"recursion_limit": 50}
    )

    if final_data_prep_state.get("status") != "success":
        return {
            **state,
            "status": "failed",
            "error": final_data_prep_state.get("error", "Data Preparation Agent failed."),
            "logs": state.get("logs", []) + final_data_prep_state.get("logs", []),
            "artifacts": {
                **state.get("artifacts", {}),
                **final_data_prep_state.get("artifacts", {}),
            },
        }

    prepared_dataset_path = final_data_prep_state["artifacts"]["prepared_dataset_path"]

    modeling_next_input = {
        "target_column": next_input.get("target_column"),
        "task_type": next_input.get("task_type"),
        "prepared_dataset_path": prepared_dataset_path,
        "data_preparation_summary": final_data_prep_state.get("preparation_summary", {}),
        "created_features": final_data_prep_state.get("created_features", []),
    }

    return {
        **state,
        "current_dataset_path": prepared_dataset_path,
        "prepared_dataset_path": prepared_dataset_path,
        "next_input": modeling_next_input,
        "data_preparation": {
            "status": "success",
            "summary": final_data_prep_state.get("preparation_summary", {}),
            "created_features": final_data_prep_state.get("created_features", []),
        },
        "artifacts": {
            **state.get("artifacts", {}),
            **final_data_prep_state.get("artifacts", {}),
        },
        "logs": state.get("logs", []) + final_data_prep_state.get("logs", []),
        "status": "running",
    }

In [ ]:
from langgraph.graph import StateGraph, START, END


def build_start_test_graph():
    workflow = StateGraph(AgentState)

    # init-ноды
    workflow.add_node("dataset_finder", dataset_finder_node)
    workflow.add_node("previous_run_finder", previous_run_finder_node)
    workflow.add_node("join_init", join_init_node)

    # оркестратор
    workflow.add_node("orchestrator_agent", orchestrator_agent_node)

    # агенты пайплайна
    workflow.add_node("data_description_agent", data_description_agent_node)
    workflow.add_node("data_preparation_agent", run_data_preparation_agent_node)

    # параллельный старт
    workflow.add_edge(START, "dataset_finder")
    workflow.add_edge(START, "previous_run_finder")

    # join после двух init-нод
    workflow.add_edge(["dataset_finder", "previous_run_finder"], "join_init")

    # после init идем в оркестратор
    workflow.add_edge("join_init", "orchestrator_agent")

    # решение оркестратора
    workflow.add_conditional_edges(
        "orchestrator_agent",
        route_by_orchestrator_decision,
        {
            "run_data_description": "data_description_agent",
            "run_data_preparation": "data_preparation_agent",
            "run_modeling": END,
            "run_improvement": END,
            "run_reporting": END,
            "retry_current_step": END,
            "use_fallback": END,
            "finish": END,
            "fail": END,
        },
    )

    # после data_description возвращаемся в оркестратор,
    # чтобы он выбрал следующий шаг: run_data_preparation
    workflow.add_edge("data_description_agent", "orchestrator_agent")

    # после data_preparation пока останавливаем тест
    workflow.add_edge("data_preparation_agent", END)

    return workflow.compile()

: 

In [ ]:
start_test_app = build_start_test_graph()

final_state = start_test_app.invoke(
    create_initial_state(),
    {"recursion_limit": 100}
)

print(final_state.get("status"))
print(final_state.get("error"))
print(final_state.get("artifacts"))
print(final_state.get("current_dataset_path"))


========== START STAGE: basic_overview ==========

RAW RESPONSE:
{
  "stage": "basic_overview",
  "status": "success",
  "result": {
    "n_rows": 15000,
    "n_columns": 36,
    "columns": [
      "id",
      "description",
      "amenities",
      "property_type",
      "room_type",
      "accommodates",
      "bathrooms",
      "bedrooms",
      "beds",
      "host_since",
      "host_location",
      "host_response_time",
      "host_response_rate",
      "host_acceptance_rate",
      "host_is_superhost",
      "host_listings_count",
      "host_total_listings_count",
      "latitude",
      "longitude",
      "city",
      "has_availability",
      "availability_30",
      "availability_60",
      "availability_90",
      "availability_365",
      "number_of_reviews",
      "number_of_reviews_ltm",
      "number_of_reviews_l30d",
      "first_review",
      "last_review",
      "review_scores_rating",
      "review_scores_cleanliness",
      "review_scores_location",
      "revie

: 

In [ ]:
for i, log in enumerate(final_state.get("logs", []), start=1):
    print(f"\n--- Log {i} ---")
    print(json.dumps(log, ensure_ascii=False, indent=2))


--- Log 1 ---
{
  "agent": "dataset_finder",
  "status": "success",
  "skipped": false,
  "summary": "Dataset found: ./rent_predictions/airbnb_train_fe_15000.csv",
  "decisions": {
    "agent_response": "./rent_predictions/airbnb_train_fe_15000.csv",
    "fallback_used": false,
    "fallback_result": null
  },
  "artifacts": {},
  "next_input": {
    "dataset_path": "./rent_predictions/airbnb_train_fe_15000.csv"
  },
  "reason": null
}

--- Log 2 ---
{
  "agent": "previous_run_finder",
  "status": "success",
  "skipped": true,
  "summary": "No previous run was found.",
  "decisions": {
    "agent_response": "{\n  \"latest_run_path\": null,\n  \"metrics_path\": null,\n  \"best_model_path\": null\n}",
    "parsed": {
      "latest_run_path": null,
      "metrics_path": null,
      "best_model_path": null
    },
    "fallback_used": false,
    "fallback_result": null
  },
  "artifacts": {},
  "next_input": {
    "previous_run": {
      "exists": false,
      "path": null
    },
    "prev

: 

In [ ]:
final_state

{'run_id': '91d60b40-214a-4e5a-a07f-922838f28230',
 'dataset_path': './rent_predictions/airbnb_train_fe_15000.csv',
 'target_column': 'price',
 'task_type': 'regression',
 'schema': {'numeric': ['accommodates',
   'bathrooms',
   'bedrooms',
   'beds',
   'host_listings_count',
   'host_total_listings_count',
   'latitude',
   'longitude',
   'availability_30',
   'availability_60',
   'availability_90',
   'availability_365',
   'number_of_reviews',
   'number_of_reviews_ltm',
   'number_of_reviews_l30d',
   'review_scores_rating',
   'review_scores_cleanliness',
   'review_scores_location',
   'review_scores_value',
   'reviews_per_month'],
  'categorical': ['property_type',
   'room_type',
   'host_location',
   'host_response_time',
   'host_response_rate',
   'host_acceptance_rate',
   'city'],
  'text': ['description', 'amenities'],
  'datetime': ['host_since', 'first_review', 'last_review'],
  'boolean_like': ['host_is_superhost', 'has_availability'],
  'id': ['id'],
  'target':

: 

In [ ]:
print("current_step:", final_state.get("current_step"))
print("next_action:", final_state.get("next_action"))
print("status:", final_state.get("status"))
print("error:", final_state.get("error"))

print("prepared_dataset_path:", final_state.get("prepared_dataset_path"))
print("artifacts prepared:", final_state.get("artifacts", {}).get("prepared_dataset_path"))

print("next_input:", final_state.get("next_input"))

current_step: orchestrator_agent
next_action: run_tabular_preparation
status: running
error: None
prepared_dataset_path: None
artifacts prepared: artifacts/data_preparation/prepared_dataset.csv
next_input: None


: 

In [ ]:
for item in final_state.get("orchestration_history", []):
    print(item.get("decision"), "->", item.get("selected_agent"))

run_data_description -> data_description_agent
run_tabular_preparation -> tabular_preparation_agent
run_data_description -> data_description_agent
run_tabular_preparation -> tabular_preparation_agent


: 

## Делаем Modeling

In [49]:
class ModelingState(TypedDict, total=False):
    prepared_dataset_path: str
    current_dataset_path: str
    target_column: str
    task_type: str

    model_jobs: list[dict[str, Any]]
    model_results: Annotated[list[dict[str, Any]], operator.add]

    model_planning: dict[str, Any]
    model_comparison: dict[str, Any]
    tune_best_model: dict[str, Any]
    final_modeling: dict[str, Any]
    modeling_summary: dict[str, Any]

    artifacts: dict[str, Any]
    logs: Annotated[list[dict[str, Any]], operator.add]
    error: str
    status: str

In [50]:
def run_stage_with_llm_modeling(stage_name: str, task: str) -> dict:
    print(f"\n========== START MODELING STAGE: {stage_name} ==========")

    try:
        result = modeling_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        print("\nGENERATED PYTHON CODE:")
        code_found = False

        for message in result["messages"]:
            tool_calls = getattr(message, "tool_calls", None)

            if not tool_calls and isinstance(message, dict):
                tool_calls = message.get("tool_calls")

            if tool_calls:
                for tool_call in tool_calls:
                    tool_name = tool_call.get("name")
                    args = tool_call.get("args", {})

                    if tool_name == "python_interpreter_tool":
                        code = args.get("code")

                        if code:
                            code_found = True
                            print("\n" + "-" * 60)
                            print(code)
                            print("-" * 60)

        if not code_found:
            print("No Python code tool call found.")

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END MODELING STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END MODELING STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise

In [51]:
from pathlib import Path
import json


def modeling_plan_workers_node(state: ModelingState) -> ModelingState:
    prepared_dataset_path = state["prepared_dataset_path"]
    target_column = state["target_column"]
    task_type = state["task_type"]

    task = f"""
Stage: plan_model_workers

Prepared dataset path:
{prepared_dataset_path}

Target column:
{target_column}

Task type:
{task_type}

Use python_interpreter_tool if compact dataset inspection is needed.

Your task:
Create a modeling plan with several independent model workers.

The workers will be executed in parallel later.

Rules:
- Choose models suitable for the task type.
- For regression, choose from:
  LinearRegression, Ridge, Lasso, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor.
- For classification, choose from:
  LogisticRegression, RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier.
- Choose 3 to 5 models.
- Each worker must train exactly one model.
- Do not train models in this stage.
- Do not modify the dataset.
- Return only compact JSON.

Return JSON only:
{{
  "stage": "plan_model_workers",
  "status": "success",
  "result": {{
    "task_type": "{task_type}",
    "target_column": "{target_column}",
    "prepared_dataset_path": "{prepared_dataset_path}",
    "model_jobs": [
      {{
        "job_id": "model_1",
        "model_name": "LinearRegression",
        "model_type": "linear",
        "reason": "baseline regression model"
      }}
    ],
    "primary_metric": "rmse_or_f1",
    "selection_rule": "lower RMSE is better for regression, higher F1 is better for classification"
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_modeling("plan_model_workers", task)
        model_planning = parsed.get("result", {})
        model_jobs = model_planning.get("model_jobs", [])

        artifacts_dir = Path("artifacts/modeling/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        planning_path = artifacts_dir / "plan_model_workers.json"

        with open(planning_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            **state,
            "model_planning": model_planning,
            "model_jobs": model_jobs,
            "model_results": [],
            "artifacts": {
                **state.get("artifacts", {}),
                "model_planning_path": str(planning_path),
            },
            "logs": [{
                "agent": "modeling_agent",
                "stage": "plan_model_workers",
                "status": "success",
                "summary": f"Planned {len(model_jobs)} model workers.",
                "artifacts": {
                    "model_planning_path": str(planning_path),
                },
            }],
            "status": "running",
        }

    except Exception as e:
        return {
            **state,
            "error": str(e),
            "logs": [{
                "agent": "modeling_agent",
                "stage": "plan_model_workers",
                "status": "failed",
                "summary": "Model worker planning failed.",
                "reason": str(e),
            }],
            "status": "failed",
        }

In [52]:
from langgraph.types import Send


def send_model_jobs(state: ModelingState):
    return [
        Send(
            "train_model_worker",
            {
                **state,
                "model_job": job,
            }
        )
        for job in state.get("model_jobs", [])
    ]

In [53]:
from pathlib import Path
import json


def modeling_train_model_worker_node(state: ModelingState) -> ModelingState:
    prepared_dataset_path = state["prepared_dataset_path"]
    target_column = state["target_column"]
    task_type = state["task_type"]
    model_job = state["model_job"]

    job_id = model_job["job_id"]
    model_name = model_job["model_name"]

    model_artifact_path = f"artifacts/modeling/models/{job_id}_{model_name}.joblib"
    metrics_path = f"artifacts/modeling/metrics/{job_id}_{model_name}_metrics.json"

    task = f"""
Stage: train_model_worker

This is one parallel model worker.

Prepared dataset path:
{prepared_dataset_path}

Target column:
{target_column}

Task type:
{task_type}

Model job:
{json.dumps(model_job, ensure_ascii=False, indent=2)}

Model artifact path:
{model_artifact_path}

Metrics path:
{metrics_path}

Use python_interpreter_tool to write and execute short Python code.

Your task:
Train exactly one model specified in model_job.

Rules:
- Read the prepared dataset.
- Split data into train and test sets.
- Use test_size=0.2 and random_state=42.
- Do not use target column as a feature.
- Train only the requested model.
- For regression, calculate:
  MAE, RMSE, R2.
- For classification, calculate:
  accuracy, precision, recall, f1.
- Save the trained model to the requested model artifact path using joblib.
- Save metrics JSON to the requested metrics path.
- Do not modify the prepared dataset.
- Do not train other models.
- Return only compact JSON.

Critical anti-hallucination rules:
- The final JSON must use exactly the metrics calculated by the executed Python code.
- Do not invent or approximate metrics.
- Do not invent n_train or n_test.
- After executing code, read the metrics JSON file and use those values in the final answer.
- Metric keys must be lowercase: mae, rmse, r2 for regression.
- The values in the final JSON must match the saved metrics file.

Supported model mapping:
Regression:
- LinearRegression
- Ridge
- Lasso
- RandomForestRegressor
- GradientBoostingRegressor
- ExtraTreesRegressor

Classification:
- LogisticRegression
- RandomForestClassifier
- GradientBoostingClassifier
- ExtraTreesClassifier

Return JSON only:
{{
  "stage": "train_model_worker",
  "status": "success",
  "result": {{
    "job_id": "{job_id}",
    "model_name": "{model_name}",
    "task_type": "{task_type}",
    "model_artifact_path": "{model_artifact_path}",
    "metrics_path": "{metrics_path}",
    "metrics": {{}},
    "n_train": null,
    "n_test": null,
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_modeling(f"train_model_worker_{job_id}", task)
        result = parsed.get("result", {})

        new_log = {
            "agent": "modeling_agent",
            "stage": "train_model_worker",
            "status": "success",
            "summary": f"Model worker completed: {model_name}.",
            "artifacts": {
                "model_artifact_path": model_artifact_path,
                "metrics_path": metrics_path,
            },
        }

        return {
            "model_results": [result],
            "logs": [new_log],
        }

    except Exception as e:
        failed_result = {
            "job_id": job_id,
            "model_name": model_name,
            "task_type": task_type,
            "status": "failed",
            "error": str(e),
            "metrics": {},
            "model_artifact_path": None,
            "metrics_path": None,
        }

        new_log = {
            "agent": "modeling_agent",
            "stage": "train_model_worker",
            "status": "failed",
            "summary": f"Model worker failed: {model_name}.",
            "reason": str(e),
        }

        return {
            "model_results": [failed_result],
            "logs": [new_log],
        }

In [54]:
from pathlib import Path
import json


def modeling_compare_models_node(state: ModelingState) -> ModelingState:
    model_results = state.get("model_results", [])
    task_type = state["task_type"]

    artifacts_dir = Path("artifacts/modeling/stages")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    comparison_path = artifacts_dir / "compare_models.json"

    successful_results = [
        r for r in model_results
        if r.get("metrics") and r.get("model_artifact_path")
    ]

    if not successful_results:
        comparison = {
            "stage": "compare_models",
            "status": "failed",
            "result": {
                "best_model_name": None,
                "best_model_path": None,
                "reason": "No successful model results.",
                "all_results": model_results,
            },
        }

        with open(comparison_path, "w", encoding="utf-8") as f:
            json.dump(comparison, f, ensure_ascii=False, indent=2)

        return {
            **state,
            "model_comparison": comparison["result"],
            "error": "No successful model results.",
            "artifacts": {
                **state.get("artifacts", {}),
                "model_comparison_path": str(comparison_path),
            },
            "logs": [{
                "agent": "modeling_agent",
                "stage": "compare_models",
                "status": "failed",
                "summary": "No successful model results.",
            }],
            "status": "failed",
        }

    if task_type == "regression":
        best = min(
            successful_results,
            key=lambda r: r.get("metrics", {}).get("rmse", float("inf"))
        )
        primary_metric = "rmse"
        direction = "lower_is_better"
    else:
        best = max(
            successful_results,
            key=lambda r: r.get("metrics", {}).get("f1", float("-inf"))
        )
        primary_metric = "f1"
        direction = "higher_is_better"

    comparison_result = {
        "best_job_id": best.get("job_id"),
        "best_model_name": best.get("model_name"),
        "best_model_path": best.get("model_artifact_path"),
        "best_metrics": best.get("metrics"),
        "primary_metric": primary_metric,
        "direction": direction,
        "all_results": model_results,
    }

    comparison = {
        "stage": "compare_models",
        "status": "success",
        "result": comparison_result,
    }

    with open(comparison_path, "w", encoding="utf-8") as f:
        json.dump(comparison, f, ensure_ascii=False, indent=2)

    return {
        **state,
        "model_comparison": comparison_result,
        "artifacts": {
            **state.get("artifacts", {}),
            "model_comparison_path": str(comparison_path),
            "best_model_path": best.get("model_artifact_path"),
        },
        "logs": [{
            "agent": "modeling_agent",
            "stage": "compare_models",
            "status": "success",
            "summary": f"Best model selected: {best.get('model_name')}.",
            "artifacts": {
                "model_comparison_path": str(comparison_path),
                "best_model_path": best.get("model_artifact_path"),
            },
        }],
        "status": "running",
    }

In [55]:
from pathlib import Path
import json


def modeling_tune_best_model_node(state: ModelingState) -> ModelingState:
    prepared_dataset_path = state["prepared_dataset_path"]
    target_column = state["target_column"]
    task_type = state["task_type"]

    comparison = state.get("model_comparison", {})
    best_model_name = comparison.get("best_model_name")

    tuned_model_path = f"artifacts/modeling/tuned_models/tuned_{best_model_name}.joblib"
    tuned_metrics_path = f"artifacts/modeling/tuned_models/tuned_{best_model_name}_metrics.json"

    task = f"""
Stage: tune_best_model

Prepared dataset path:
{prepared_dataset_path}

Target column:
{target_column}

Task type:
{task_type}

Best model from previous comparison:
{json.dumps(comparison, ensure_ascii=False, indent=2)}

Tuned model artifact path:
{tuned_model_path}

Tuned metrics path:
{tuned_metrics_path}

Use python_interpreter_tool to write and execute short Python code.

Your task:
Tune hyperparameters only for the best model selected in the previous stage.

Rules:
- Read the prepared dataset.
- Split data into train and test sets with test_size=0.2 and random_state=42.
- Do not use the target column as a feature.
- Tune only the best model type.
- Use GridSearchCV or RandomizedSearchCV.
- Use 3-fold cross-validation.
- Use lightweight parameter grids to keep execution fast.
- Do not perform data preparation.
- Do not train other model types.
- Save the tuned best estimator to the requested tuned model artifact path using joblib.
- Save metrics and best_params to the requested tuned metrics path.
- Return only compact JSON.

Regression tuning:
- For Ridge, tune alpha.
- For Lasso, tune alpha.
- For RandomForestRegressor, tune n_estimators, max_depth, min_samples_split.
- For GradientBoostingRegressor, tune n_estimators, learning_rate, max_depth.
- For ExtraTreesRegressor, tune n_estimators, max_depth, min_samples_split.
- Scoring metric: neg_root_mean_squared_error.
- Final test metrics: mae, rmse, r2.

Classification tuning:
- For LogisticRegression, tune C.
- For RandomForestClassifier, tune n_estimators, max_depth, min_samples_split.
- For GradientBoostingClassifier, tune n_estimators, learning_rate, max_depth.
- For ExtraTreesClassifier, tune n_estimators, max_depth, min_samples_split.
- Scoring metric: f1_weighted.
- Final test metrics: accuracy, precision, recall, f1.

Critical anti-hallucination rules:
- The final JSON must use exactly the metrics calculated by the executed Python code.
- Do not invent or approximate metrics.
- Do not invent best_params.
- After executing code, read the tuned metrics JSON file and use those values in the final answer.
- The values in the final JSON must match the saved tuned metrics file.

Use these lightweight grids:

Regression:
- Ridge: {{"alpha": [0.1, 1.0, 10.0, 50.0]}}
- Lasso: {{"alpha": [0.0001, 0.001, 0.01], "max_iter": [3000]}}
- RandomForestRegressor: {{"n_estimators": [50, 100], "max_depth": [8, 12, None], "min_samples_split": [2, 5]}}
- GradientBoostingRegressor: {{"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}}
- ExtraTreesRegressor: {{"n_estimators": [50, 100], "max_depth": [8, 12, None], "min_samples_split": [2, 5]}}

Classification:
- LogisticRegression: {{"C": [0.1, 1.0, 10.0], "max_iter": [1000]}}
- RandomForestClassifier: {{"n_estimators": [50, 100], "max_depth": [8, 12, None], "min_samples_split": [2, 5]}}
- GradientBoostingClassifier: {{"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}}
- ExtraTreesClassifier: {{"n_estimators": [50, 100], "max_depth": [8, 12, None], "min_samples_split": [2, 5]}}

Return JSON only:
{{
  "stage": "tune_best_model",
  "status": "success",
  "result": {{
    "best_model_name": "{best_model_name}",
    "tuned_model_path": "{tuned_model_path}",
    "tuned_metrics_path": "{tuned_metrics_path}",
    "best_params": {{}},
    "metrics": {{}},
    "n_train": null,
    "n_test": null,
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_modeling("tune_best_model", task)
        tune_result = parsed.get("result", {})

        # Надёжно читаем реальные метрики из файла, а не доверяем тексту LLM
        if Path(tuned_metrics_path).exists():
            with open(tuned_metrics_path, "r", encoding="utf-8") as f:
                saved_payload = json.load(f)

            tune_result["best_params"] = saved_payload.get("best_params", tune_result.get("best_params", {}))
            tune_result["metrics"] = saved_payload.get("metrics", tune_result.get("metrics", {}))
            tune_result["n_train"] = saved_payload.get("n_train", tune_result.get("n_train"))
            tune_result["n_test"] = saved_payload.get("n_test", tune_result.get("n_test"))

        tune_result["best_model_name"] = best_model_name
        tune_result["tuned_model_path"] = tuned_model_path
        tune_result["tuned_metrics_path"] = tuned_metrics_path

        artifacts_dir = Path("artifacts/modeling/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        tune_stage_path = artifacts_dir / "tune_best_model.json"

        with open(tune_stage_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "stage": "tune_best_model",
                    "status": "success",
                    "result": tune_result,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        new_log = {
            "agent": "modeling_agent",
            "stage": "tune_best_model",
            "status": "success",
            "summary": f"Hyperparameter tuning completed for {best_model_name}.",
            "artifacts": {
                "tuned_model_path": tuned_model_path,
                "tuned_metrics_path": tuned_metrics_path,
                "tune_best_model_path": str(tune_stage_path),
            },
        }

        return {
            **state,
            "tune_best_model": tune_result,
            "artifacts": {
                **state.get("artifacts", {}),
                "tuned_model_path": tuned_model_path,
                "tuned_metrics_path": tuned_metrics_path,
                "tune_best_model_path": str(tune_stage_path),
            },
            "logs": [new_log],
            "status": "running",
        }

    except Exception as e:
        new_log = {
            "agent": "modeling_agent",
            "stage": "tune_best_model",
            "status": "failed",
            "summary": "Hyperparameter tuning failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [56]:
from pathlib import Path
import json
import shutil


def modeling_final_node(state: ModelingState) -> ModelingState:
    comparison = state.get("model_comparison", {})
    tuning = state.get("tune_best_model", {})

    best_model_name = comparison.get("best_model_name")

    # Если tuning прошёл успешно, берём tuned model.
    # Если нет — fallback на лучшую baseline-модель.
    best_model_path = (
        tuning.get("tuned_model_path")
        or comparison.get("best_model_path")
    )

    best_metrics = (
        tuning.get("metrics")
        or comparison.get("best_metrics", {})
    )

    best_params = tuning.get("best_params", {})

    final_model_path = "artifacts/modeling/best_model.joblib"
    final_metrics_path = "artifacts/modeling/final_metrics.json"
    modeling_summary_path = "artifacts/modeling/modeling_summary.json"

    Path("artifacts/modeling").mkdir(parents=True, exist_ok=True)

    if not best_model_path:
        new_log = {
            "agent": "modeling_agent",
            "stage": "final_modeling",
            "status": "failed",
            "summary": "Final modeling failed because best model path is missing.",
            "reason": "best_model_path is None",
        }

        return {
            **state,
            "error": "best_model_path is None",
            "logs": state.get("logs", []) + [new_log],
            "status": "failed",
        }

    shutil.copy(best_model_path, final_model_path)

    modeling_summary = {
        "status": "success",
        "task_type": state.get("task_type"),
        "target_column": state.get("target_column"),
        "prepared_dataset_path": state.get("prepared_dataset_path"),
        "models_tested": [
            r.get("model_name")
            for r in state.get("model_results", [])
            if r.get("model_name")
        ],
        "best_model_name": best_model_name,
        "best_model_path": final_model_path,
        "best_metrics": best_metrics,
        "tuning_used": bool(tuning),
        "best_params": best_params,
        "baseline_best_metrics": comparison.get("best_metrics", {}),
        "tuned_metrics": tuning.get("metrics", {}),
        "model_comparison": comparison,
        "tuning_result": tuning,
    }

    with open(final_metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "best_model_name": best_model_name,
                "best_metrics": best_metrics,
                "best_params": best_params,
                "tuning_used": bool(tuning),
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(modeling_summary_path, "w", encoding="utf-8") as f:
        json.dump(modeling_summary, f, ensure_ascii=False, indent=2)

    new_log = {
        "agent": "modeling_agent",
        "stage": "final_modeling",
        "status": "success",
        "summary": f"Final best model saved: {best_model_name}.",
        "artifacts": {
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
        },
    }

    return {
        **state,
        "final_modeling": {
            "best_model_name": best_model_name,
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
            "tuning_used": bool(tuning),
            "best_params": best_params,
        },
        "modeling_summary": modeling_summary,
        "artifacts": {
            **state.get("artifacts", {}),
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
        },
        "logs": state.get("logs", []) + [new_log],
        "status": "success",
        "error": None,
    }

In [57]:
from langgraph.graph import StateGraph, START, END


modeling_graph = StateGraph(ModelingState)

modeling_graph.add_node("plan_model_workers", modeling_plan_workers_node)
modeling_graph.add_node("train_model_worker", modeling_train_model_worker_node)
modeling_graph.add_node("compare_models", modeling_compare_models_node)
modeling_graph.add_node("tune_best_model", modeling_tune_best_model_node)
modeling_graph.add_node("final_modeling", modeling_final_node)

modeling_graph.add_edge(START, "plan_model_workers")

modeling_graph.add_conditional_edges(
    "plan_model_workers",
    send_model_jobs,
    ["train_model_worker"],
)

modeling_graph.add_edge("train_model_worker", "compare_models")
modeling_graph.add_edge("compare_models", "tune_best_model")
modeling_graph.add_edge("tune_best_model", "final_modeling")
modeling_graph.add_edge("final_modeling", END)

modeling_app = modeling_graph.compile()

In [63]:
modeling_state = {
    "prepared_dataset_path": 'artifacts/data_preparation/prepared_dataset.csv',
    "current_dataset_path": 'artifacts/data_preparation/prepared_dataset.csv',
    "target_column": 'price',
    "task_type": 'regression',
    "model_jobs": [],
    "model_results": [],
    "model_planning": {},
    "model_comparison": {},
    "final_modeling": {},
    "modeling_summary": {},
    "artifacts": {},
    "logs": [],
    "status": "running",
}

In [64]:
modeling_state

{'prepared_dataset_path': 'artifacts/data_preparation/prepared_dataset.csv',
 'current_dataset_path': 'artifacts/data_preparation/prepared_dataset.csv',
 'target_column': 'price',
 'task_type': 'regression',
 'model_jobs': [],
 'model_results': [],
 'model_planning': {},
 'model_comparison': {},
 'final_modeling': {},
 'modeling_summary': {},
 'artifacts': {},
 'logs': [],
 'status': 'running'}

In [65]:
final_modeling_state = modeling_app.invoke(
    modeling_state,
    {"recursion_limit": 80}
)

print("Status:", final_modeling_state.get("status"))
print("Error:", final_modeling_state.get("error"))

print("Best model:", final_modeling_state.get("modeling_summary", {}).get("best_model_name"))
print("Best params:", final_modeling_state.get("modeling_summary", {}).get("best_params"))
print("Best metrics:", final_modeling_state.get("modeling_summary", {}).get("best_metrics"))

print("Artifacts:", final_modeling_state.get("artifacts"))


========== START MODELING STAGE: plan_model_workers ==========

GENERATED PYTHON CODE:
No Python code tool call found.

RAW RESPONSE:
{
  "stage": "plan_model_workers",
  "status": "success",
  "result": {
    "task_type": "regression",
    "target_column": "price",
    "prepared_dataset_path": "artifacts/data_preparation/prepared_dataset.csv",
    "model_jobs": [
      {
        "job_id": "model_1",
        "model_name": "Ridge",
        "model_type": "linear",
        "reason": "Linear model with L2 regularization to handle potential multicollinearity"
      },
      {
        "job_id": "model_2",
        "model_name": "RandomForestRegressor",
        "model_type": "ensemble",
        "reason": "Robust non-linear ensemble model based on bagging"
      },
      {
        "job_id": "model_3",
        "model_name": "GradientBoostingRegressor",
        "model_type": "ensemble",
        "reason": "Powerful boosting ensemble for capturing complex patterns"
      },
      {
        "job_id

In [58]:
def run_modeling_agent_node(state: AgentState) -> AgentState:
    print("=== RUN MODELING AGENT ===")

    prepared_dataset_path = (
        state.get("prepared_dataset_path")
        or state.get("artifacts", {}).get("prepared_dataset_path")
        or state.get("next_input", {}).get("prepared_dataset_path")
    )

    target_column = (
        state.get("target_column")
        or state.get("next_input", {}).get("target_column")
    )

    task_type = (
        state.get("task_type")
        or state.get("next_input", {}).get("task_type")
    )

    modeling_state: ModelingState = {
        "prepared_dataset_path": prepared_dataset_path,
        "current_dataset_path": prepared_dataset_path,
        "target_column": target_column,
        "task_type": task_type,
        "model_jobs": [],
        "model_results": [],
        "model_planning": {},
        "model_comparison": {},
        "tune_best_model": {},
        "final_modeling": {},
        "modeling_summary": {},
        "artifacts": {},
        "logs": [],
        "status": "running",
    }

    final_modeling_state = modeling_app.invoke(
        modeling_state,
        {"recursion_limit": 80}
    )

    merged_artifacts = {
        **state.get("artifacts", {}),
        **final_modeling_state.get("artifacts", {}),
    }

    merged_logs = (
        state.get("logs", [])
        + [
            {
                "agent": "modeling_agent",
                "status": "success"
                if final_modeling_state.get("status") == "success"
                else "failed",
                "skipped": False,
                "summary": "Modeling agent completed."
                if final_modeling_state.get("status") == "success"
                else "Modeling agent failed.",
                "decisions": {
                    "best_model_name": final_modeling_state.get("modeling_summary", {}).get("best_model_name"),
                    "best_metrics": final_modeling_state.get("modeling_summary", {}).get("best_metrics"),
                    "best_params": final_modeling_state.get("modeling_summary", {}).get("best_params"),
                },
                "artifacts": final_modeling_state.get("artifacts", {}),
                "next_input": {
                    "best_model_path": merged_artifacts.get("best_model_path"),
                    "modeling_summary_path": merged_artifacts.get("modeling_summary_path"),
                },
                "reason": final_modeling_state.get("error"),
                "subgraph_logs": final_modeling_state.get("logs", []),
            }
        ]
    )

    if final_modeling_state.get("status") != "success":
        return {
            **state,
            "status": "failed",
            "error": final_modeling_state.get("error", "Modeling Agent failed."),
            "artifacts": merged_artifacts,
            "logs": merged_logs,
            "current_step": "modeling_agent",
        }

    modeling_summary = final_modeling_state.get("modeling_summary", {})

    return {
        **state,
        "modeling_summary": modeling_summary,
        "artifacts": merged_artifacts,
        "logs": merged_logs,
        "current_step": "modeling_agent",
        "next_action": None,
        "orchestration_decision": None,
        "status": "running",
        "error": None,
    }

## Последняя хуйня

In [59]:
from pathlib import Path
import json


def run_reporting_agent(state: AgentState) -> AgentState:
    print("=== RUN REPORTING AGENT ===")

    final_report_path = "artifacts/reporting/final_report.md"
    reporting_summary_path = "artifacts/reporting/reporting_summary.json"

    task = f"""
Create a final Markdown report for the ML project.

Dataset path:
{state.get("dataset_path")}

Target column:
{state.get("target_column") or state.get("next_input", {}).get("target_column")}

Task type:
{state.get("task_type") or state.get("next_input", {}).get("task_type")}

Data summary:
{json.dumps(state.get("data_summary", {}), ensure_ascii=False, indent=2)}

Schema:
{json.dumps(state.get("schema", {}), ensure_ascii=False, indent=2)}

Data preparation summary:
{json.dumps(state.get("data_preparation", {}), ensure_ascii=False, indent=2)}

Modeling summary:
{json.dumps(state.get("modeling_summary", {}), ensure_ascii=False, indent=2)}

Artifacts:
{json.dumps(state.get("artifacts", {}), ensure_ascii=False, indent=2)}

Save the Markdown report to:
{final_report_path}

Save a compact JSON reporting summary to:
{reporting_summary_path}

The Markdown report must include:
1. Project title
2. Business task
3. Dataset overview
4. Target column and task type
5. Feature groups from schema
6. Data quality summary
7. Data preparation steps
8. Created features
9. Models tested
10. Best model
11. Best metrics
12. Hyperparameter tuning result if available
13. Business interpretation
14. Final artifacts

Return JSON only:
{{
  "agent": "reporting_agent",
  "status": "success",
  "skipped": false,
  "summary": "Final report was created and saved.",
  "artifacts": {{
    "final_report_path": "{final_report_path}",
    "reporting_summary_path": "{reporting_summary_path}"
  }},
  "next_input": {{}},
  "reason": null
}}
"""

    try:
        result = reporting_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        raw = result["messages"][-1].content.strip()
        parsed = extract_json(raw)

        Path("artifacts/reporting").mkdir(parents=True, exist_ok=True)

        reporting_stage_path = "artifacts/reporting/reporting_agent_result.json"

        with open(reporting_stage_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "reporting_agent",
            "status": "success",
            "skipped": False,
            "summary": "Final report was created and saved.",
            "artifacts": {
                "final_report_path": final_report_path,
                "reporting_summary_path": reporting_summary_path,
                "reporting_stage_path": reporting_stage_path,
            },
            "next_input": {},
            "reason": None,
        }

        return {
            **state,
            "final_report": parsed,
            "artifacts": {
                **state.get("artifacts", {}),
                "final_report_path": final_report_path,
                "reporting_summary_path": reporting_summary_path,
                "reporting_stage_path": reporting_stage_path,
            },
            "logs": state.get("logs", []) + [new_log],
            "current_step": "reporting_agent",
            "next_action": None,
            "orchestration_decision": None,
            "status": "success",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "reporting_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Final report creation failed.",
            "reason": str(e),
        }

        return {
            **state,
            "error": str(e),
            "logs": state.get("logs", []) + [new_log],
            "current_step": "reporting_agent",
            "status": "failed",
        }

## Проверяем нах все

In [60]:
from langgraph.graph import StateGraph, START, END


def build_full_graph():
    workflow = StateGraph(AgentState)

    # init-ноды
    workflow.add_node("dataset_finder", dataset_finder_node)
    workflow.add_node("previous_run_finder", previous_run_finder_node)
    workflow.add_node("join_init", join_init_node)

    # оркестратор
    workflow.add_node("orchestrator_agent", orchestrator_agent_node)

    # агенты пайплайна
    workflow.add_node("data_description_agent", data_description_agent_node)
    workflow.add_node("data_preparation_agent", run_data_preparation_agent_node)
    workflow.add_node("modeling_agent", run_modeling_agent_node)
    workflow.add_node("reporting_agent", run_reporting_agent)

    # параллельный старт
    workflow.add_edge(START, "dataset_finder")
    workflow.add_edge(START, "previous_run_finder")

    # join после init
    workflow.add_edge(["dataset_finder", "previous_run_finder"], "join_init")

    # после init идём в оркестратор
    workflow.add_edge("join_init", "orchestrator_agent")

    # маршрутизация оркестратора
    workflow.add_conditional_edges(
        "orchestrator_agent",
        route_by_orchestrator_decision,
        {
            "run_data_description": "data_description_agent",
            "run_data_preparation": "data_preparation_agent",
            "run_modeling": "modeling_agent",
            "run_improvement": "modeling_agent",
            "run_reporting": "reporting_agent",
            "retry_current_step": "orchestrator_agent",
            "use_fallback": "orchestrator_agent",
            "finish": END,
            "fail": END,

            # временная совместимость, если orchestrator всё ещё вернёт старые action
            "run_tabular_preparation": "data_preparation_agent",
            "run_text_preparation": "data_preparation_agent",
            "merge_features": "data_preparation_agent",
        },
    )

    # после каждого агента возвращаемся к оркестратору
    workflow.add_edge("data_description_agent", "orchestrator_agent")
    workflow.add_edge("data_preparation_agent", "orchestrator_agent")
    workflow.add_edge("modeling_agent", "orchestrator_agent")

    # reporting — финальная стадия
    workflow.add_edge("reporting_agent", END)

    return workflow.compile()

In [61]:
full_app = build_full_graph()

initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "price",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

final_state = full_app.invoke(
    initial_state,
    {"recursion_limit": 200}
)


========== START STAGE: basic_overview ==========

RAW RESPONSE:
{
  "stage": "basic_overview",
  "status": "success",
  "result": {
    "n_rows": 15000,
    "n_columns": 36,
    "columns": [
      "id",
      "description",
      "amenities",
      "property_type",
      "room_type",
      "accommodates",
      "bathrooms",
      "bedrooms",
      "beds",
      "host_since",
      "host_location",
      "host_response_time",
      "host_response_rate",
      "host_acceptance_rate",
      "host_is_superhost",
      "host_listings_count",
      "host_total_listings_count",
      "latitude",
      "longitude",
      "city",
      "has_availability",
      "availability_30",
      "availability_60",
      "availability_90",
      "availability_365",
      "number_of_reviews",
      "number_of_reviews_ltm",
      "number_of_reviews_l30d",
      "first_review",
      "last_review",
      "review_scores_rating",
      "review_scores_cleanliness",
      "review_scores_location",
      "revie

In [69]:
data = pd.read_csv('artifacts/data_preparation/prepared_dataset.csv')
data['price'].head()

0    143.0
1     71.0
2    500.0
3    126.0
4     94.0
Name: price, dtype: float64